# GRH : ce qu'on peut calculer — les briques vérifiables d'une annonce

Accrétion du carnet [12-zeros-simples-ligne-critique](12-zeros-simples-ligne-critique.ipynb), dans la série [Riemann 200](README.md) (EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403), point 3 de l'issue [#27589](https://github.com/jsboige/CoursIA/issues/27589)).

Le carnet 12 montre qu'une mesure finie ne dit rien d'une limite : cent zéros sur la ligne critique ne font pas l'hypothèse de Riemann. Celui-ci prend le problème par un autre bout. Une note du 20 novembre 2026 annonce une démonstration ; au lieu de commenter l'annonce, on calcule les **briques qu'un lecteur peut vérifier lui-même**, et l'on regarde ce que chacune donne à petite échelle : un déterminant sur une courbe, des sommes de von Mangoldt, une longueur de chaîne. La chaîne automorphe qui porte l'argument annoncé est **citée, jamais reproduite**.

**Statut épistémique (à lire avant tout le reste).** La démonstration annoncée par L. Alpöge dans *Remark on the Riemann Hypothesis* (note du 20 novembre 2026, 4 pages) est une **annonce de l'auteur, non relue par les pairs**. Ce carnet ne reproduit aucun énoncé formel, et il ne lit pas l'archive de preuves qui accompagne la note : le dénombrement de cette archive et la recherche d'axiomes y ont été faits une fois, ailleurs dans le dépôt, et sont cités ici comme *Rapportés*. **Rien de l'argument annoncé ne doit être présenté comme établi.**

Trois textes sont lus, et il faut les distinguer. Les trois sont archivés au gisement partagé (GDrive `MyIA/IA/Bibliographie IA`, rayon `NumberTheory`) et cités depuis leur copie :

- **la note de l'auteur** (20 novembre 2026) : une annonce, avec une chronologie de travail. Tout ce qui en vient est marqué *Annoncé* ;
- **un guide d'architecture**, publié en ligne sous le nom de projet `riemann-rev` (11 pages, dont les relevés s'arrêtent au 20 novembre 2026 à 23 h 51 UTC) : une lecture tierce de la note et de l'archive. Son auteur prévient que ses sections 4 à 7 sont « read here from the Lean source and from the announcement, not re-derived ». Les formules des parties B et C sont citées d'après lui ;
- **une déclaration conjointe** de l'équipe de développement de Lean et de l'initiative Mathlib (21 novembre 2026) : la communauté rapporte avoir comparé l'énoncé démontré à celui de Mathlib avec deux outils indépendants, et avoir fait vérifier la preuve exportée par plusieurs vérificateurs. Une certification formelle porte sur un fichier de preuve : elle n'est pas une relecture par les pairs, et elle ne dit rien de la compréhension des idées.

**Statuts, déclarés une fois pour toutes** (convention des carnets 13 et 14, augmentée d'un quatrième statut) : *Mesuré* = calculé dans une sortie de ce carnet ; *Cité* = lu dans une source archivée, nommée entre parenthèses ; *Annoncé* = issu de la note de l'auteur, sans vérification ; *Rapporté* = lu dans un inventaire du dépôt, hors de ce carnet — ce dernier statut est propre à ce carnet. Chaque partie **pré-enregistre ses hypothèses avant de mesurer**, et le verdict — confirmée ou réfutée — reste dans les sorties.

## Plan

1. **La racine carrée, côté courbe** (partie A) : des points d'une courbe sur un corps fini, une dérivation, et un déterminant calculé avec sa forme close.
2. **Les sommes de von Mangoldt et leur ordre de grandeur** (partie B) : une somme lissée $G(y)$, ses oscillations lues sur les cent premiers zéros de $\zeta$, et une expérience construite qui montre à quoi sert une majoration sur des fenêtres.
3. **La longueur $\sqrt{X}$ en échelle finie** (partie C) : une factorielle et ses valuations, un caractère de Dirichlet, et le compromis entre une chaîne trop courte et une chaîne trop longue.
4. **Limites et bilan honnête** : ce que les mesures prouvent, ce qu'elles suggèrent, ce qu'elles ne touchent pas.
5. **Exercices** : quatre, répartis dans le texte au fil des parties, à compléter.
6. **Raccords au dépôt**.

Calcul en **stdlib seule**, sans aucune dépendance externe : les zéros de $\zeta$ sont cherchés par la formule d'Euler–Maclaurin. Les dix hypothèses H1 à H10 sont **pré-enregistrées** dans le texte, chacune avant la cellule qui la mesure.

In [1]:
# Dependances : stdlib pure (bisect, cmath, itertools, math, random, fractions).
# Aucun mpmath dans ce carnet : les zeros de zeta sont cherches par Euler-Maclaurin.
import bisect
import cmath
import itertools
import math
import random
from fractions import Fraction
from math import comb, factorial, sqrt

print('stdlib seule, kernel python3', __import__('sys').version.split()[0])

stdlib seule, kernel python3 3.13.16


## Partie A — La racine carrée, côté courbe

L'auteur décrit ainsi sa source (*Annoncé*, note du 20 novembre 2026, p. 1) : « In Stepanov’s argument resolving Riemann over function fields one uses a divisor involving multiplicity. This argument is a descendant of Stepanov’s, but multiplicity is hard to find over the integers, corresponding to something like differentiating whole numbers. »

Le guide raconte la même chose en une phrase (*Cité*, guide, ouverture, p. 1) : « For curves over a finite field 𝔽, Stepanov obtained it from an auxiliary function vanishing to high order at the rational points: comparing zeros with the degree bounds the number of points from above, and upper bounds over all finite extensions of 𝔽 suffice. » Trois mots y portent tout : *auxiliary function*, *vanishing to high order*, *comparing zeros with the degree*. Cette partie construit ces trois objets sur des courbes assez petites pour que tout se calcule, et vérifie le budget qui les relie : une fonction non nulle a au plus autant de zéros, comptés avec multiplicité, que l'ordre de son pôle.

Elle ne traite **que** le cas d'une courbe sur un corps fini. La transcription du côté des entiers, qui est l'objet de l'annonce, n'en est pas donnée ici.

### A.1 — Une courbe, ses points, et les extensions

On pose $y^2 = f(x)$ avec $f(x) = x^3 + a x + b$ sur $\mathbb{F}_p$, et l'on compte. Le théorème de Hasse borne la trace $t = p + 1 - N_1$, où $N_1$ compte les points affines et le point à l'infini : $|t| \le 2\sqrt{p}$. La même trace commande les extensions : $N_n = p^n + 1 - S_n$, avec $S_0 = 2$, $S_1 = t$ et $S_n = t S_{n-1} - p S_{n-2}$. Le guide signale que, dans l'argument annoncé, une tour cyclotomique « plays the part of the finite extensions of 𝔽 in Stepanov’s proof » (*Cité*, guide, § 4) : c'est le geste que l'on répète ici à petite échelle.

**H1 (pré-enregistrée).** Sur six courbes $y^2 = x^3 + a x + b$ ($p \le 97$), on aura $|t| \le 2\sqrt{p}$. Sur les quatre premières, en comptant les points sur place dans $\mathbb{F}_{p^n}$ pour $n = 2$ puis $n = 3$, on retrouvera exactement $N_n = p^n + 1 - S_n$.

Le carnet mesure cette inégalité ; il ne démontre pas le théorème de Hasse.

In [2]:
def evalue(g, x, p):
    """Valeur en x du polynome g (coefficients croissants), modulo p (schema de Horner)."""
    r = 0
    for c in reversed(g):
        r = (r * x + c) % p
    return r


def points_affines(p, a, b):
    """Points (x, y) de y^2 = x^3 + a x + b a coordonnees dans F_p."""
    carres = {}                                  # on range les carres par leur valeur
    for y in range(p):
        carres.setdefault((y * y) % p, []).append(y)
    pts = []
    for x in range(p):
        v = (x * x * x + a * x + b) % p
        for y in carres.get(v, []):
            pts.append((x, y))
    return pts


def sans_racine(p, n):
    """Premier polynome monique de degre n sans racine dans F_p (donc irreductible pour n <= 3)."""
    g = [0] * (n + 1)
    g[n] = 1
    for q in range(p ** n):                      # les n coefficients bas decrivent un nombre < p^n
        c = q
        for i in range(n):
            g[i] = c % p
            c //= p
        if all(evalue(g, x, p) != 0 for x in range(p)):
            return tuple(g)
    raise ValueError('aucun polynome sans racine')


class Extension:
    """F_{p^n} = F_p[w] / (g(w)) ; un element est code par ses n coefficients, ecrits en base p."""

    def __init__(self, p, n):
        self.p, self.n = p, n
        self.g = sans_racine(p, n)

    def vers_poly(self, u):
        c = []
        for _ in range(self.n):
            c.append(u % self.p)
            u //= self.p
        return c

    def depuis_poly(self, c):
        u = 0
        for x in reversed(c):
            u = u * self.p + x % self.p
        return u

    def add(self, u, v):
        return self.depuis_poly([(x + y) % self.p for x, y in zip(self.vers_poly(u), self.vers_poly(v))])

    def mul(self, u, v):
        prod = [0] * (2 * self.n - 1)
        for i, x in enumerate(self.vers_poly(u)):
            for j, y in enumerate(self.vers_poly(v)):
                prod[i + j] = (prod[i + j] + x * y) % self.p
        for d in range(2 * self.n - 2, self.n - 1, -1):     # reduction modulo g, du haut vers le bas
            c = prod[d]
            for i in range(self.n + 1):
                prod[d - self.n + i] = (prod[d - self.n + i] - c * self.g[i]) % self.p
        return self.depuis_poly(prod[: self.n])


def points_extension(F, a, b):
    """Nombre de points affines de la meme courbe, vue sur F = F_{p^n}."""
    carres = {}                                   # table des carres de F, comptes une seule fois
    for y in range(F.p ** F.n):
        y2 = F.mul(y, y)
        carres[y2] = carres.get(y2, 0) + 1
    total = 0
    for x in range(F.p ** F.n):
        x3 = F.mul(F.mul(x, x), x)
        ax = F.depuis_poly([(a * c) % F.p for c in F.vers_poly(x)])
        v = F.add(F.add(x3, ax), b % F.p)
        total += carres.get(v, 0)
    return total


def suite_s(p, t):
    """S_0 = 2, S_1 = t, S_n = t S_{n-1} - p S_{n-2}, pour n = 0 .. 3."""
    S = [2, t]
    for n in range(2, 4):
        S.append(t * S[-1] - p * S[-2])
    return S


echantillon = [(5, 1, 1), (7, 1, 1), (7, 2, 1), (11, 1, 2), (13, 1, 2), (97, 3, 4)]
print('   p  (a, b)     N_1      t    |t| / (2 sqrt p)')
for (p, a, b) in echantillon:
    t = p + 1 - len(points_affines(p, a, b)) - 1
    print(f'  {p:>3}  ({a}, {b})   {p + 1 - t:>5}   {t:>+4}   {abs(t) / (2 * sqrt(p)):.3f}')
h1a = all(abs(p + 1 - len(points_affines(p, a, b)) - 1) <= 2 * sqrt(p) for (p, a, b) in echantillon)

h1b = True
print('extensions : N_n compte sur place, contre p^n + 1 - S_n')
for (p, a, b) in echantillon[:4]:
    t = p + 1 - len(points_affines(p, a, b)) - 1
    S = suite_s(p, t)
    for n in (2, 3):
        F = Extension(p, n)
        Nn = points_extension(F, a, b) + 1
        h1b = h1b and (Nn == p ** n + 1 - S[n])
        print(f'  p = {p:>2}, (a, b) = ({a}, {b}), n = {n}, g = {F.g} : N_n = {Nn:>4}, p^n + 1 - S_n = {p ** n + 1 - S[n]:>4}')
print('H1 :', 'CONFIRMEE' if (h1a and h1b) else 'REFUTEE')

   p  (a, b)     N_1      t    |t| / (2 sqrt p)
    5  (1, 1)       9     -3   0.671
    7  (1, 1)       5     +3   0.567
    7  (2, 1)       5     +3   0.567
   11  (1, 2)      16     -4   0.603
   13  (1, 2)      12     +2   0.277
   97  (3, 4)      88    +10   0.508
extensions : N_n compte sur place, contre p^n + 1 - S_n
  p =  5, (a, b) = (1, 1), n = 2, g = (2, 0, 1) : N_n =   27, p^n + 1 - S_n =   27
  p =  5, (a, b) = (1, 1), n = 3, g = (1, 1, 0, 1) : N_n =  108, p^n + 1 - S_n =  108
  p =  7, (a, b) = (1, 1), n = 2, g = (1, 0, 1) : N_n =   55, p^n + 1 - S_n =   55
  p =  7, (a, b) = (1, 1), n = 3, g = (2, 0, 0, 1) : N_n =  380, p^n + 1 - S_n =  380
  p =  7, (a, b) = (2, 1), n = 2, g = (1, 0, 1) : N_n =   55, p^n + 1 - S_n =   55
  p =  7, (a, b) = (2, 1), n = 3, g = (2, 0, 0, 1) : N_n =  380, p^n + 1 - S_n =  380
  p = 11, (a, b) = (1, 2), n = 2, g = (1, 0, 1) : N_n =  128, p^n + 1 - S_n =  128
  p = 11, (a, b) = (1, 2), n = 3, g = (4, 1, 0, 1) : N_n = 1264, p^n + 1 - S_n = 126

### Lecture du résultat

**H1 : CONFIRMÉE.** Sur les six courbes, le rapport $|t| / (2\sqrt{p})$ reste sous 0,67 : la borne de Hasse tient, avec de la marge. Et le comptage sur place dans $\mathbb{F}_{p^2}$ puis $\mathbb{F}_{p^3}$ redonne, aux huit essais, le nombre $p^n + 1 - S_n$ que la seule trace annonçait : pour $p = 5$, où $t = -3$, on compte 27 puis 108 points, c'est-à-dire $25 + 1 + 1$ et $125 + 1 - 18$.

Deux leçons pour la suite. D'abord, **une seule trace commande tous les comptages des extensions** : c'est le sens de la phrase du guide sur la tour cyclotomique, qui « plays the part of the finite extensions of 𝔽 ». Ensuite, **rien ici n'est une démonstration** : l'inégalité de Hasse est un théorème, et le carnet n'en voit que six instances. Six exemples ne distingueraient pas la borne $2\sqrt{p}$ d'une borne un peu plus large.

### A.2 — Une dérivation, et la multiplicité

Sur un corps fini, dériver « whole numbers » n'a pas de sens. Sur la courbe, cela en a un : il existe une seule dérivation $D$ de l'anneau des fonctions qui vérifie $D(x) = f(x)$. L'équation $y^2 = f(x)$ l'impose entièrement, puisque $D(y^2) = 2y\,D(y)$ doit égaler $D(f(x)) = f'(x)\,f(x)$ : donc $D(y) = \frac{f'(x)}{2}\,y$. Pour une fonction $u = A(x) + y\,B(x)$, on lit directement $D(A + yB) = fA' + y\,\big(fB' + \frac{f'}{2}B\big)$.

La multiplicité, elle, se lit sur les dérivées usuelles en $x$. En un point $P = (x_0, y_0)$ d'ordonnée non nulle, la $j$-ème itérée $D^j u$ d'une fonction $u(x)$ est une combinaison triangulaire de $u', u'', \dots, u^{(j)}$, dont le coefficient diagonal vaut $f(x_0)^j \ne 0$. Donc $u, Du, \dots, D^{k-1}u$ s'annulent en $P$ si et seulement si $x_0$ est racine d'ordre au moins $k$ de $u$ : **s'annuler à l'ordre $k$ pour $D$, c'est s'annuler à l'ordre $k$ pour la variable**.

**H2 (pré-enregistrée).** Sur trois courbes ($p = 7$, $13$, $31$), la règle de Leibniz $D(uv) = D(u)\,v + u\,D(v)$ ne sera jamais en défaut sur 300 produits de fonctions aléatoires. Sur $\mathbb{F}_7$, pour tous les polynômes non nuls de degré au plus 5 et tous les points d'ordonnée non nulle, l'ordre de $x_0$ comme racine de $u$ égalera le nombre d'itérées de $D$ qui s'annulent en $P$.

In [3]:
def ptrim(u, p):
    """Reduit les coefficients modulo p et supprime ceux de tete nuls."""
    u = [x % p for x in u]
    while u and u[-1] == 0:
        u.pop()
    return u


def padd(u, v, p):
    """Somme de deux polynomes a coefficients dans F_p (coefficients croissants)."""
    n = max(len(u), len(v))
    return ptrim([(u[i] if i < len(u) else 0) + (v[i] if i < len(v) else 0) for i in range(n)], p)


def pmul(u, v, p):
    """Produit de deux polynomes a coefficients dans F_p."""
    w = [0] * (len(u) + len(v) - 1) if u and v else []
    for i, x in enumerate(u):
        for j, y in enumerate(v):
            w[i + j] = (w[i + j] + x * y) % p
    return ptrim(w, p)


def pder(u, p):
    """Derivee formelle d'un polynome."""
    return ptrim([i * u[i] for i in range(1, len(u))], p)


class Courbe:
    """y^2 = f(x), f = x^3 + a x + b, sur F_p ; une fonction est un couple (A, B) pour A + y B."""

    def __init__(self, p, a, b):
        self.p, self.a, self.b = p, a, b
        self.f = ptrim([b, a, 0, 1], p)
        self.fp = pder(self.f, p)
        self.inv2 = pow(2, p - 2, p)              # inverse de 1/2 modulo p

    def produit(self, u, v):
        """(A1 + y B1)(A2 + y B2) = A1 A2 + f B1 B2 + y (A1 B2 + A2 B1), par y^2 = f."""
        (A1, B1), (A2, B2) = u, v
        return (padd(pmul(A1, A2, self.p), pmul(self.f, pmul(B1, B2, self.p), self.p), self.p),
                padd(pmul(A1, B2, self.p), pmul(A2, B1, self.p), self.p))

    def valeur(self, u, P):
        (x, y) = P
        return (evalue(u[0], x, self.p) + y * evalue(u[1], x, self.p)) % self.p

    def derive(self, u):
        """D(A + y B) = f A' + y (f B' + (f'/2) B)."""
        A, B = u
        return (pmul(self.f, pder(A, self.p), self.p),
                padd(pmul(self.f, pder(B, self.p), self.p),
                     [self.inv2 * c % self.p for c in pmul(self.fp, B, self.p)], self.p))


def aleatoire(C, rng, degre):
    """Polynome de degre < degre a coefficients aleatoires dans F_p."""
    return ptrim([rng.randrange(C.p) for _ in range(degre)], C.p)


def somme(u, v, C):
    """Somme de deux fonctions sur la courbe C."""
    return (padd(u[0], v[0], C.p), padd(u[1], v[1], C.p))


h2a = True
for (p, a, b) in [(7, 1, 1), (13, 1, 2), (31, 3, 4)]:
    C = Courbe(p, a, b)
    rng = random.Random(p)                        # graine fixee : la sortie est reproductible
    ecarts = 0
    for _ in range(300):
        u = (aleatoire(C, rng, 6), aleatoire(C, rng, 5))
        v = (aleatoire(C, rng, 6), aleatoire(C, rng, 5))
        if C.derive(C.produit(u, v)) != somme(C.produit(C.derive(u), v), C.produit(u, C.derive(v)), C):
            ecarts += 1
    y2 = C.produit(([], [1]), ([], [1]))          # y fois y, reduit modulo y^2 = f
    coherent = ecarts == 0 and y2 == (C.f, [])    # Leibniz sans defaut, et y^2 = f sans reste en y
    h2a = h2a and coherent
    print(f'p = {p:>2} : Leibniz en defaut sur {ecarts} produit sur 300 ; y^2 = (f, 0) : {y2 == (C.f, [])}')


def ordre_racine(u, x, p):
    """Multiplicite de x comme racine du polynome u non nul (0 si x n'est pas racine)."""
    k, v = 0, list(u)
    while v and evalue(v, x, p) == 0:
        v = pder(v, p)
        k += 1
    return k


def premier_non_nul(u, P, C):
    """Plus petit j tel que (D^j u)(P) soit non nul, pour u polynome et P d'ordonnee non nulle."""
    j, uu = 0, (list(u), [])
    while C.valeur(uu, P) == 0 and uu != ([], []):
        uu = C.derive(uu)
        j += 1
    return j


C = Courbe(7, 1, 1)
points = [P for P in points_affines(7, 1, 1) if P[1] != 0]
ecarts = testes = 0
for degre in range(6):                            # tous les polynomes non nuls de degre <= 5 sur F_7
    for coeffs in itertools.product(range(7), repeat=degre + 1):
        u = ptrim(list(coeffs), 7)
        if u:
            testes += 1
            for P in points:
                ecarts += ordre_racine(u, P[0], 7) != premier_non_nul(u, P, C)
print(f'F_7 : {testes} polynomes non nuls de degre <= 5, {len(points)} points d ordonnee non nulle, ecarts = {ecarts}')
print('H2 :', 'CONFIRMEE' if (h2a and ecarts == 0) else 'REFUTEE')

p =  7 : Leibniz en defaut sur 0 produit sur 300 ; y^2 = (f, 0) : True


p = 13 : Leibniz en defaut sur 0 produit sur 300 ; y^2 = (f, 0) : True
p = 31 : Leibniz en defaut sur 0 produit sur 300 ; y^2 = (f, 0) : True


F_7 : 137250 polynomes non nuls de degre <= 5, 4 points d ordonnee non nulle, ecarts = 0
H2 : CONFIRMEE


### Lecture du résultat

**H2 : CONFIRMÉE.** Sur les trois courbes, aucun des 300 produits ne met Leibniz en défaut, et $y^2$ se réduit bien à $f(x)$, sans reste en $y$. Sur $\mathbb{F}_7$, les 137 250 polynômes non nuls de degré au plus 5, confrontés aux 4 points d'ordonnée non nulle, donnent zéro écart : l'ordre de la racine et le nombre d'itérées de $D$ qui s'annulent coïncident toujours.

C'est l'équivalent, sur une courbe, de ce que l'annonce appelle « differentiating whole numbers » : **la dérivation $D$ voit la multiplicité**. Trois limites, toutefois. La dérivation existe sur toute courbe $y^2 = f(x)$ ; ce qui est propre aux courbes à multiplication complexe, c'est le comportement de ses itérées — la chronologie de l'auteur note, pour une telle courbe et un premier $p$ inerte, « D^p ≡ 0 on a CM curve for p inert in K, D the invariant derivation » (*Annoncé*). L'égalité des ordres, ensuite, est une identité, démontrable par récurrence sur $j$, que le calcul illustre sans établir. Et rien de tout cela, enfin, ne se transpose tel quel aux entiers.

### A.3 — Le déterminant

Pour construire une fonction auxiliaire nulle à l'ordre $k$ en $s$ points $P_1, \dots, P_s$, on cherche une combinaison linéaire $u = \sum_{c} \lambda_c\,x^c$ des monômes $x^0, \dots, x^{ks}$ qui vérifie les $ks$ conditions $(D^j u)(P_i) = 0$, pour $j < k$. Il y a $ks + 1$ inconnues et $ks$ équations : le noyau est de dimension au moins 1 dès que la matrice $ks \times ks$ des $ks$ premières colonnes est inversible. Son déterminant est la brique que la chronologie de l'auteur appelle « 10/6/26: Stepanov determinant. » (*Annoncé*). La note ne dit pas ce que ce déterminant est ; celui qu'on calcule ici est le déterminant des conditions de jets, un déterminant de Vandermonde confluent, brique classique de la méthode.

On impose aux points des abscisses deux à deux distinctes : un point et son opposé ont les mêmes jets en $x$. La matrice des $(D^j x^c)(P_i)$ a même déterminant que la matrice d'entrée $f(x_i)^j\,(x^c)^{(j)}(x_i)$, puisque la $j$-ème itérée est une combinaison triangulaire des dérivées usuelles, de coefficient diagonal $f^j$. Et cette dernière a une forme close, produit de trois facteurs :

$$\det = \prod_i f(x_i)^{k(k-1)/2} \;\cdot\; \prod_{j < k} (j!)^s \;\cdot\; \prod_{i < i'} (x_{i'} - x_i)^{k^2}.$$

**H3 (pré-enregistrée).** Sur quatre courbes ($p = 7$, $11$, $13$, $17$), pour $k \le 4$ et $s \le 5$, avec des abscisses distinctes et des ordonnées non nulles, le déterminant calculé par élimination sera **égal** à la forme close, et **non nul**, sur toute la grille. En un point $P$ et son opposé $-P$, à $k = 1$, il sera nul.

In [4]:
def determinant(M, p):
    """Determinant d'une matrice carree a coefficients dans F_p, par elimination."""
    M = [[x % p for x in ligne] for ligne in M]
    n = len(M)
    det = 1
    for col in range(n):
        pivot = next((r for r in range(col, n) if M[r][col]), None)    # premier pivot non nul
        if pivot is None:
            return 0
        if pivot != col:
            M[col], M[pivot] = M[pivot], M[col]
            det = -det
        inv = pow(M[col][col], p - 2, p)
        det = det * M[col][col] % p
        for r in range(col + 1, n):
            fact = M[r][col] * inv % p
            for c in range(col, n):
                M[r][c] = (M[r][c] - fact * M[col][c]) % p
    return det % p


def matrice_jet(points, k, C, nb_colonnes=None):
    """Lignes (i, j) pour j < k, colonnes c : entree f(x_i)^j (x^c)^{(j)}(x_i)."""
    s = len(points)
    M = []
    for (x, y) in points:
        for j in range(k):
            fxj = pow(evalue(C.f, x, C.p), j, C.p)
            ligne = []
            for c in range(k * s if nb_colonnes is None else nb_colonnes):
                coef = 1                                    # c (c - 1) ... (c - j + 1)
                for l in range(j):
                    coef *= c - l
                ligne.append(0 if c < j else fxj * coef % C.p * pow(x, c - j, C.p) % C.p)
            M.append(ligne)
    return M


def forme_close(points, k, C):
    """Produit des f(x_i)^{k(k-1)/2}, des (j!)^s et des (x_{i'} - x_i)^{k^2}."""
    s = len(points)
    p = C.p
    prod = 1
    for (x, y) in points:
        prod = prod * pow(evalue(C.f, x, p), k * (k - 1) // 2, p) % p
    for j in range(k):
        prod = prod * pow(factorial(j), s, p) % p
    for i in range(s):
        for i2 in range(i + 1, s):
            prod = prod * pow(points[i2][0] - points[i][0], k * k, p) % p
    return prod % p


def abscisses_distinctes(C, s, rng):
    """s points d'ordonnee non nulle, d'abscisses deux a deux distinctes."""
    un_par_abscisse = {}
    for P in points_affines(C.p, C.a, C.b):
        if P[1] != 0:
            un_par_abscisse.setdefault(P[0], P)
    return rng.sample(list(un_par_abscisse.values()), s)


ecarts = testes = 0
rng = random.Random(2026)                        # graine fixee : la grille parcourue est reproductible
for (p, a, b) in [(7, 1, 1), (11, 1, 2), (13, 1, 2), (17, 2, 2)]:
    C = Courbe(p, a, b)
    nb_max = len({x for (x, y) in points_affines(p, a, b) if y != 0})
    for k in (1, 2, 3, 4):
        for s in range(1, min(5, nb_max) + 1):
            points = abscisses_distinctes(C, s, rng)
            d, fc = determinant(matrice_jet(points, k, C), p), forme_close(points, k, C)
            testes += 1
            ecarts += (d != fc) or (fc == 0)
print(f'grille k <= 4, s <= 5, quatre courbes : {testes} determinants calcules, ecarts a la forme close = {ecarts}')

C = Courbe(7, 1, 1)
P = next(Q for Q in points_affines(7, 1, 1) if Q[1] != 0)
moins_P = (P[0], (-P[1]) % 7)                    # le point de meme abscisse, d'ordonnee opposee
print('P =', P, '; -P =', moins_P, '; k = 1 : determinant =', determinant(matrice_jet([P, moins_P], 1, C), 7),
      '; forme close =', forme_close([P, moins_P], 1, C))
print('H3 :', 'CONFIRMEE' if ecarts == 0 else 'REFUTEE')

grille k <= 4, s <= 5, quatre courbes : 68 determinants calcules, ecarts a la forme close = 0
P = (0, 1) ; -P = (0, 6) ; k = 1 : determinant = 0 ; forme close = 0
H3 : CONFIRMEE


### Lecture du résultat

**H3 : CONFIRMÉE.** Sur les 68 déterminants de la grille, l'élimination et la forme close donnent le même élément de $\mathbb{F}_p$, jamais nul : la matrice des conditions de jets est inversible dès que les abscisses sont distinctes. Et pour $P = (0, 1)$ et $-P = (0, 6)$ à $k = 1$, le déterminant est nul, comme la forme close l'annonce : les deux points ont la même abscisse, et le facteur $(x_{i'} - x_i)$ s'annule. Un point et son opposé ne peuvent pas porter deux conditions indépendantes sur un polynôme en $x$.

La forme close dit aussi **pourquoi** la matrice est inversible : $f(x_i) \ne 0$ parce que l'ordonnée est non nulle, $j!$ est non nul parce que $j < k \le 4 < p$ sur les quatre courbes, et les différences d'abscisses sont non nulles par hypothèse. Le calcul ne démontre pas l'identité ; il la vérifie sur 68 instances, et c'est assez pour l'utiliser sans crainte dans la suite.

### A.4 — La fonction auxiliaire et le budget

Avec une colonne de plus que d'équations, la matrice $ks \times (ks + 1)$ a un noyau de dimension exactement 1 : le vecteur $(\lambda_0, \dots, \lambda_{ks})$ de ses cofacteurs. La fonction $u = \sum_c \lambda_c x^c$ s'annule alors à l'ordre $k$ en chaque $P_i$, et son coefficient de tête $\lambda_{ks}$ est le déterminant de la section carrée, donc non nul : $u$ est de degré exactement $ks$.

Vient la comparaison des zéros avec le degré. Sur la courbe, l'ordre se mesure par rapport au pôle à l'infini. La fonction $x$ y a un pôle d'ordre 2, la fonction $y$ un pôle d'ordre 3, donc $A(x) + yB(x)$ a un pôle d'ordre $\max(2\deg A,\ 2\deg B + 3)$. Une fonction non nulle a au plus autant de zéros, comptés avec multiplicité, que l'ordre de son pôle.

**Pré-enregistrement.** Avant de mesurer, on annonce trois résultats : (i) le vecteur des cofacteurs annule la matrice, et $u$ a l'ordre 2 en chacun des quatre points ; (ii) sur 2 000 fonctions aléatoires de $y^2 = x^3 + x + 2$ sur $\mathbb{F}_{11}$, le nombre de zéros ne dépasse **jamais** l'ordre du pôle ; (iii) pour la fonction auxiliaire, les deux nombres sont **égaux**.

In [5]:
def fonction_auxiliaire(points, k, C):
    """Noyau de la matrice des jets a k s + 1 colonnes, par les cofacteurs de sa premiere ligne."""
    s = len(points)
    M = matrice_jet(points, k, C, nb_colonnes=k * s + 1)
    lam = []
    for c in range(k * s + 1):                    # mineur obtenu en retirant la colonne c
        mineur = [ligne[:c] + ligne[c + 1:] for ligne in M]
        lam.append(((-1) ** c * determinant(mineur, C.p)) % C.p)
    return lam, M


C = Courbe(11, 1, 2)
k = 2                                            # ordre impose en chaque point
points = abscisses_distinctes(C, 4, random.Random(1))
s = len(points)
lam, M = fonction_auxiliaire(points, k, C)
image = [sum(M[r][c] * lam[c] for c in range(k * s + 1)) % C.p for r in range(k * s)]
u = ptrim(lam, C.p)                               # la fonction auxiliaire : sum lambda_c x^c
print('points :', points)
print('produit matrice fois lambda :', image)
print('u =', u)
print('degre de u =', len(u) - 1, '= k s ; coefficient de tete = determinant de la matrice carree =',
      determinant([ligne[:-1] for ligne in M], C.p))
print('ordre de chaque P_i comme racine de u :', [ordre_racine(u, P[0], C.p) for P in points])


def multiplicite(C, u, P):
    """Ordre du zero de u = A + y B en un point P d'ordonnee non nulle (0 si u(P) != 0)."""
    j, v = 0, (list(u[0]), list(u[1]))
    while C.valeur(v, P) == 0 and v != ([], []):
        v = C.derive(v)
        j += 1
    return j


def ordre_du_pole(u):
    """Ordre du pole a l'infini de A + y B (x a un pole d'ordre 2, y d'ordre 3)."""
    return max(2 * (len(u[0]) - 1), 2 * (len(u[1]) - 1) + 3)


def zeros_sur_la_courbe(C, u):
    """Somme des multiplicites de u sur les points affines a coordonnees dans F_p."""
    total = 0
    for (x, y) in points_affines(C.p, C.a, C.b):
        if y != 0:
            total += multiplicite(C, u, (x, y))
        elif C.valeur(u, (x, y)) == 0:             # ordonnee nulle : le parametre local est y
            A, B = u
            ordre_A = ordre_racine(A, x, C.p) if A else 10 ** 9
            ordre_B = ordre_racine(B, x, C.p) if B else 10 ** 9
            total += min(2 * ordre_A, 2 * ordre_B + 1)
    return total


rng = random.Random(2026)
egalites = depassements = 0
for _ in range(2000):
    v = (aleatoire(C, rng, 4), aleatoire(C, rng, 3))
    if v == ([], []):
        continue
    zeros, pole = zeros_sur_la_courbe(C, v), ordre_du_pole(v)
    depassements += zeros > pole                    # le budget ne doit jamais etre depasse
    egalites += zeros == pole
print('2000 fonctions aleatoires : zeros > ordre du pole dans', depassements, 'cas ; egalite dans', egalites, 'cas')

print('fonction auxiliaire : zeros sur F_11 avec multiplicite =', zeros_sur_la_courbe(C, (u, [])),
      '; ordre du pole a l infini =', ordre_du_pole((u, [])))
x_p_moins_x = [0] * (C.p + 1)
x_p_moins_x[1], x_p_moins_x[C.p] = C.p - 1, 1   # x^p - x : nulle en tout x de F_p
print('x^p - x : zeros sur F_11 =', zeros_sur_la_courbe(C, (x_p_moins_x, [])),
      '; ordre du pole = 2p =', ordre_du_pole((x_p_moins_x, [])),
      '; points affines =', len(points_affines(C.p, C.a, C.b)),
      'dont', sum(1 for (x, y) in points_affines(C.p, C.a, C.b) if y == 0), 'd ordonnee nulle')

points : [(2, 1), (8, 4), (1, 2), (9, 5)]
produit matrice fois lambda : [0, 0, 0, 0, 0, 0, 0, 0]
u = [3, 7, 7, 7, 2, 0, 3, 1, 3]
degre de u = 8 = k s ; coefficient de tete = determinant de la matrice carree = 3
ordre de chaque P_i comme racine de u : [2, 2, 2, 2]
2000 fonctions aleatoires : zeros > ordre du pole dans 0 cas ; egalite dans 10 cas
fonction auxiliaire : zeros sur F_11 avec multiplicite = 16 ; ordre du pole a l infini = 16
x^p - x : zeros sur F_11 = 18 ; ordre du pole = 2p = 22 ; points affines = 15 dont 3 d ordonnee nulle


### Lecture du résultat

Les trois annonces tiennent. Le vecteur des cofacteurs donne un produit matrice fois vecteur nul, et chacun des quatre points est racine d'ordre 2 de $u = 3 + 7x + 7x^2 + 7x^3 + 2x^4 + 3x^6 + x^7 + 3x^8$, de degré $ks = 8$ et de coefficient de tête 3, égal au déterminant de la section carrée (*Mesuré*). Sur 2 000 fonctions aléatoires, le nombre de zéros ne dépasse jamais l'ordre du pôle, et ne l'atteint que 10 fois : une fonction quelconque gaspille presque toujours son budget. La fonction auxiliaire, elle, le sature exactement : 16 zéros pour un pôle d'ordre 16.

D'où viennent ces 16, alors que l'on n'en a imposé que 8 ? De la symétrie. Une fonction de $x$ seule prend la même valeur en $P = (x_0, y_0)$ et en $-P = (x_0, -y_0)$ : s'annuler à l'ordre 2 en $P_i$, c'est s'annuler à l'ordre 2 en $-P_i$. **La moitié du budget est dépensée par symétrie**, et c'est la même raison qui interdisait, en A.3, de prendre un point et son opposé. Enfin $x^p - x$, nulle en tout point à abscisse dans $\mathbb{F}_{11}$, a 18 zéros pour un pôle d'ordre 22 : simple aux 12 points d'ordonnée non nulle, double aux 3 points d'ordonnée nulle, où $y$ est la variable locale et $x - x_0$ d'ordre 2.

**Ce que cette partie établit et ce qu'elle ne touche pas.** Établi, à l'échelle des exemples : la dérivation voit la multiplicité, la matrice des jets est inversible sous la seule condition d'abscisses distinctes, et le compte des zéros ne dépasse pas l'ordre du pôle. Non traité : le pas suivant de la méthode de Stepanov, où l'on choisit les paramètres pour que le budget des zéros borne le nombre de points. Et rien ici ne franchit le passage aux entiers, qui est l'objet de l'annonce.

### Exercice 1 — Refaire le déterminant sur une autre courbe

Sur $y^2 = x^3 + x + 2$ sur $\mathbb{F}_{13}$, deux points d'abscisses distinctes sont tirés avec la graine 7. À vous de calculer le déterminant de la matrice des jets pour $k = 2$, puis la forme close, et de vérifier l'égalité. Les deux fonctions de la partie A.3 s'appliquent directement.

In [6]:
# Exercice 1 - a completer : determinant des jets et forme close sur F_13, pour k = 2.
# Etape 1 : construisez la matrice des jets des deux points pour k = 2 (matrice_jet).
# Etape 2 : calculez son determinant modulo 13 (determinant).
# Etape 3 : calculez la forme close (forme_close) et comparez les deux.
# Indice : les deux fonctions sont definies en A.3 ; l'exemple de A.4 montre l'appel.
C_exo = Courbe(13, 1, 2)
points_exo = abscisses_distinctes(C_exo, 2, random.Random(7))
det_exo = None    # TODO etudiant : determinant de la matrice des jets, pour k = 2
forme_exo = None  # TODO etudiant : forme close du meme determinant
print('points :', points_exo)
print('Exercice a completer : determinant =', det_exo, '; forme close =', forme_exo)

points : [(6, 4), (2, 5)]
Exercice a completer : determinant = None ; forme close = None


## Partie B — Les sommes de von Mangoldt et leur ordre de grandeur

Sur les entiers, la fonction auxiliaire n'est plus une fonction sur une courbe : c'est une somme sur les nombres premiers. Le guide pose, pour $y \ge 1$ (*Cité*, guide, § 2) :

$$G(y) = \sum_{n \le y} \Lambda(n)\Big(\frac{y}{n} - 1\Big) - y(\log y - 1 - \gamma),$$

où $\Lambda(n)$ vaut $\log p$ si $n$ est une puissance du premier $p$, et 0 sinon. Le poids $\frac{y}{n} - 1$ est une rampe : « it vanishes at $n = y$ and grows as $n$ decreases ». Le dernier terme est le centrage : « it is what makes the transform below regular at $s = 1$ ».

Le transformé de Mellin de $G$ se prolonge au-delà de son domaine de convergence, et ses seules singularités sont des pôles simples aux zéros $\rho$ de $\zeta$. Le guide en tire la formule explicite (*Cité*, guide, § 2) :

> « Read backwards, this is the explicit formula G(y) = log 2π − Σ_ρ y^ρ/(ρ(ρ−1)) + O(y^{−2}), in which ρ runs over the zeros of ζ in the critical strip, with multiplicity. Hence RH gives |G(y) − log 2π| ≤ 0.0462√y + O(y^{−2}), the constant being 2 + γ − log 4π. Numerically one finds |G(y) − log 2π| < 0.023√y for 10 ≤ y ≤ 10^7. »

Il faut lire cette phrase dans le bon sens. **L'hypothèse de Riemann y est la prémisse, pas la conclusion.** Si tous les zéros étaient sur la ligne $\Re s = \tfrac12$, on aurait $|y^\rho| = y^{1/2}$ pour chacun, et la somme des $\tfrac{1}{|\rho(\rho - 1)|}$ donnerait la constante $2 + \gamma - \log 4\pi$. Rien de cette partie ne vérifie ni ne met en doute l'hypothèse. Elle vérifie les **énoncés numériques** que le guide associe à $G$, et elle construit un faux témoin qui montre à quoi servirait une majoration sur des fenêtres.

### B.0 — Les outils : nombres de Bernoulli exacts, zêta par Euler–Maclaurin

Toute la partie, et la suivante, reposent sur trois outils que la bibliothèque standard ne fournit pas. On les construit.

- **Les nombres de Bernoulli**, en rationnels exacts, par leur relation de définition $\sum_{r = 0}^{m} \binom{m + 1}{r} B_r = 0$, avec $B_1 = -\tfrac12$. Le carnet 13 leur est en partie consacré.
- **La fonction zêta de Hurwitz** $\zeta(s, q) = \sum_{n \ge 0} (n + q)^{-s}$, par la formule d'Euler–Maclaurin : $N$ termes sommés, puis le terme intégral $\tfrac{x^{1 - s}}{s - 1}$, le terme $\tfrac{x^{-s}}{2}$, et $M$ corrections en $\frac{B_{2m}}{(2m)!}\,s(s + 1)\cdots(s + 2m - 2)\,x^{-s - 2m + 1}$, avec $x = N + q$. Pour $s$ complexe, chaque puissance est complexe.
- **La constante d'Euler** $\gamma$, par la série harmonique et ses corrections : $H_N - \log N - \tfrac{1}{2N} + \sum_m \frac{B_{2m}}{2m} N^{-2m}$.

La formule d'Euler–Maclaurin a un pôle en $s = 1$ : on ne l'y évaluera jamais (la partie C entoure ce point au lieu de le toucher). Avant de s'en servir, on contrôle l'outil sur des valeurs connues : $\zeta(2)$ contre $\pi^2/6$, puis $\zeta(3)$ et $\zeta(\tfrac12)$, et l'on calcule la constante du guide, $2 + \gamma - \log 4\pi$.

### B.1 — La fonction de compte de Tchebychev

Avant $G$, sa cousine sans poids : $\psi(x) = \sum_{n \le x} \Lambda(n)$. Le théorème des nombres premiers dit que $\psi(x) \sim x$. L'erreur $\psi(x) - x$ est la grandeur dont l'ordre de grandeur est en jeu : un théorème classique, non démontré et non vérifié ici, rattache l'hypothèse de Riemann à cette erreur, puisqu'elle équivaut à $\psi(x) = x + O(x^{1/2 + \varepsilon})$ pour tout $\varepsilon > 0$. C'est la raison du rapport à $\sqrt{x}$ dans la dernière colonne.

On crible les nombres premiers jusqu'à $10^7$, et l'on range les puissances de premiers une fois pour toutes : les sommes partielles de $\Lambda(n)$ et de $\Lambda(n)/n$ serviront aussi en B.2 et en partie C.

In [7]:
def bernoulli(n):
    """B_0, ..., B_n, en rationnels exacts, par la relation de definition (avec B_1 = -1/2)."""
    B = [Fraction(0)] * (n + 1)
    B[0] = Fraction(1)
    for m in range(1, n + 1):
        somme = Fraction(0)
        for r in range(m):
            somme += comb(m + 1, r) * B[r]
        B[m] = -somme / comb(m + 1, m)
    return B


def hurwitz(s, q, N, M, B):
    """zeta(s, q) = sum_{n >= 0} (n + q)^(-s) par Euler-Maclaurin : N termes sommes, M corrections."""
    total = sum((n + q) ** (-s) for n in range(N))       # N premiers termes, sommes tels quels
    x = N + q
    total += x ** (1 - s) / (s - 1) + x ** (-s) / 2       # terme integral, puis demi-terme de bord
    monte = s + 0j                              # produit s (s + 1) ... (s + 2m - 2), reconstruit
    for m in range(1, M + 1):
        total += float(B[2 * m]) / factorial(2 * m) * monte * x ** (-s - 2 * m + 1)
        monte *= (s + 2 * m - 1) * (s + 2 * m)
    return total


B = bernoulli(30)

N0 = 100                                        # gamma par la serie harmonique et ses corrections
harmonique = sum(1.0 / r for r in range(1, N0 + 1))
gamma = (harmonique - math.log(N0) - 1 / (2 * N0)
         + sum(float(B[2 * m]) / (2 * m) * N0 ** (-2 * m) for m in range(1, 11)))

print('B_2, B_4, ..., B_8 :', [str(b) for b in B[2:9:2]])
print('zeta(2) - pi^2 / 6 =', float(abs(hurwitz(2.0, 1.0, 10, 6, B) - math.pi ** 2 / 6)))
print('zeta(3) =', hurwitz(3.0, 1.0, 10, 6, B).real, '; zeta(1/2) =', hurwitz(0.5, 1.0, 40, 8, B).real)
print('gamma =', gamma)
print('2 + gamma - log(4 pi) =', round(2 + gamma - math.log(4 * math.pi), 5))

B_2, B_4, ..., B_8 : ['1/6', '-1/30', '1/42', '-1/30']
zeta(2) - pi^2 / 6 = 2.220446049250313e-16
zeta(3) = 1.2020569031595942 ; zeta(1/2) = -1.4603545088095855
gamma = 0.5772156649015328
2 + gamma - log(4 pi) = 0.04619


### Lecture du résultat

Les contrôles passent (*Mesuré*). L'écart entre $\zeta(2)$ et $\pi^2/6$ vaut $2{,}2 \times 10^{-16}$ : c'est l'arrondi de $\pi^2/6$, et l'outil est exact à la précision de la machine sur ce point. $\zeta(3) = 1{,}2020569$ et $\zeta(\tfrac12) = -1{,}4603545$ sont les valeurs de référence usuelles, à la précision affichée. Les nombres de Bernoulli sortent en rationnels exacts : $B_2 = \tfrac16$, $B_4 = -\tfrac1{30}$, $B_6 = \tfrac1{42}$, $B_8 = -\tfrac1{30}$.

La constante d'Euler vaut $\gamma = 0{,}5772156649$. La constante du guide, $2 + \gamma - \log 4\pi$, vaut **0,04619** : c'est le majorant de $|G(y) - \log 2\pi| / \sqrt{y}$ que la formule explicite et l'hypothèse de Riemann donneraient ensemble. Le guide annonce que l'on observe mieux, 0,023 : on y revient en B.2, après avoir regardé $\psi$.

In [8]:
LIM = 10 ** 7
marqueur = bytearray([1]) * (LIM + 1)           # crible d'Eratosthene jusqu'a 10^7
marqueur[0:2] = b'\x00\x00'
for p in range(2, int(LIM ** 0.5) + 1):
    if marqueur[p]:
        marqueur[p * p::p] = bytearray(len(range(p * p, LIM + 1, p)))
premiers = [n for n in range(2, LIM + 1) if marqueur[n]]

puissances = []                                  # couples (n = p^e, log p), tries par n
for p in premiers:
    logp, q = math.log(p), p
    while q <= LIM:
        puissances.append((q, logp))
        q *= p
puissances.sort()

indice = [n for n, logp in puissances]           # sommes partielles de Lambda(n) et Lambda(n)/n
somme_lambda = [0.0]
somme_sur_n = [0.0]
for n, logp in puissances:
    somme_lambda.append(somme_lambda[-1] + logp)
    somme_sur_n.append(somme_sur_n[-1] + logp / n)


def psi(x):
    """psi(x) = sum_{n <= x} Lambda(n)."""
    return somme_lambda[bisect.bisect_right(indice, x)]


print(len(premiers), 'nombres premiers et', len(puissances), 'puissances de premiers <= 10^7')
print('      x      psi(x) - x     (psi(x) - x) / sqrt(x)')
for k in range(3, 8):
    x = 10.0 ** k
    print(f'{x:>9.0f}   {psi(x) - x:>+11.2f}   {(psi(x) - x) / sqrt(x):>+8.4f}')

664579 nombres premiers et 665134 puissances de premiers <= 10^7
      x      psi(x) - x     (psi(x) - x) / sqrt(x)
     1000         -3.32    -0.1050
    10000        +13.40    +0.1340
   100000        +51.56    +0.1631
  1000000       -413.40    -0.4134
 10000000      -1460.60    -0.4619


### Lecture du résultat

Le crible trouve 664 579 nombres premiers inférieurs à $10^7$, le nombre tabulé : le contrôle de l'outil passe. Il y en a 665 134 en comptant les puissances de premiers, et ce sont elles qui portent $\Lambda$.

L'erreur $\psi(x) - x$ change de signe d'une ligne à l'autre et, en valeur absolue, elle grossit : de $-3{,}32$ à $x = 1000$ à $-1460{,}60$ à $x = 10^7$. Mais grossit par rapport à quoi ? Devant $x$, elle s'évanouit : $-1460$ devant dix millions, c'est moins de deux dix-millièmes. Devant $\sqrt{x}$, elle reste d'ordre constant, entre $-0{,}46$ et $+0{,}16$. **L'échelle $\sqrt{x}$ est celle où l'erreur ne disparaît pas** : c'est elle qui sépare « erreur petite » de « erreur petite comme $\sqrt{x}$ ».

### B.2 — La somme $G(y)$ et son encadrement numérique

On calcule maintenant $G(y)$ lui-même. La rampe se prête au même rangement : $\sum_{n \le y} \Lambda(n)\big(\tfrac{y}{n} - 1\big) = y \sum_{n \le y} \frac{\Lambda(n)}{n} - \sum_{n \le y} \Lambda(n)$, donc deux sommes cumulées suffisent.

**H4 (pré-enregistrée).** Sur un balayage de 2 000 points répartis géométriquement entre 10 et $10^7$, le maximum de $|G(y) - \log 2\pi| \, / \sqrt{y}$ sera **inférieur à 0,023**. Et le calcul par sommes cumulées sera confirmé, à $y = 10^6$, par une somme directe évaluée autrement : écart relatif inférieur à $10^{-8}$.

Le guide parle de l'intervalle entier ; un balayage n'est pas l'intervalle. Deux points voisins de la grille sont à 0,7 % l'un de l'autre, alors que les oscillations de $G$ ont, avec le premier zéro, une période d'un facteur 1,56 en $y$. Un dépassement entre deux points serait donc surprenant, mais le calcul ne l'exclut pas.

In [9]:
LOG2PI = math.log(2 * math.pi)


def grand_g(y):
    """G(y) = sum_{n <= y} Lambda(n) (y/n - 1) - y (log y - 1 - gamma)."""
    i = bisect.bisect_right(indice, y)
    return y * somme_sur_n[i] - somme_lambda[i] - y * (math.log(y) - 1 - gamma)


print('      y       G(y) - log 2 pi    (G(y) - log 2 pi) / sqrt(y)')
for k in range(1, 8):
    y = 10.0 ** k
    print(f'{y:>9.0f}   {grand_g(y) - LOG2PI:>+17.6f}   {(grand_g(y) - LOG2PI) / sqrt(y):>+8.5f}')

grille = [10 * 10 ** (6 * i / 1999) for i in range(2000)]      # 2000 points, repartis geometriquement
rapports = [abs(grand_g(y) - LOG2PI) / sqrt(y) for y in grille]
pire = max(range(2000), key=lambda i: rapports[i])

y_temoin = 10 ** 6                                # controle : somme directe, sans sommes cumulees
direct = (math.fsum(logp * (y_temoin / n - 1) for n, logp in puissances if n <= y_temoin)
          - y_temoin * (math.log(y_temoin) - 1 - gamma))
print(f'max sur la grille [10, 10^7] : {rapports[pire]:.5f}, atteint pres de y = {grille[pire]:.0f}')
print(f'controle a y = 10^6 : G = {grand_g(y_temoin):.9f} par sommes cumulees, {direct:.9f} par somme directe')
h4 = max(rapports) < 0.023 and abs(grand_g(y_temoin) - direct) / abs(direct) < 1e-8
print('H4 :', 'CONFIRMEE' if h4 else 'REFUTEE')

      y       G(y) - log 2 pi    (G(y) - log 2 pi) / sqrt(y)
       10           +0.022921   +0.00725
      100           -0.116190   -0.01162
     1000           -0.306964   -0.00971
    10000           -0.088562   -0.00089
   100000           +1.833585   +0.00580
  1000000           +9.202559   +0.00920
 10000000          +26.025895   +0.00823
max sur la grille [10, 10^7] : 0.02255, atteint pres de y = 320076
controle a y = 10^6 : G = 11.040436083 par sommes cumulees, 11.040436128 par somme directe
H4 : CONFIRMEE


### Lecture du résultat

**H4 : CONFIRMÉE, avec une marge mince.** Le contrôle passe d'abord : à $y = 10^6$, la somme directe et le calcul par sommes cumulées donnent $G(y) = 11{,}040436$, l'un à ...083, l'autre à ...128 près : un écart de $4{,}5 \times 10^{-10}$, qui ne vient que de l'ordre des additions (*Mesuré*).

Puis la mesure. L'écart $G(y) - \log 2\pi$ grossit en valeur absolue, de $0{,}02$ à $y = 10$ jusqu'à $26{,}0$ à $y = 10^7$. Divisé par $\sqrt{y}$, il tient entre $-0{,}0116$ et $+0{,}0092$ aux puissances de dix. Sur la grille entière, le maximum vaut **0,02255**, atteint vers $y = 320\,076$ : en dessous de 0,023, de 2 % environ. C'est aussi un peu moins de la moitié de la constante 0,04619 : l'inégalité triangulaire, qui additionne les valeurs absolues des termes de la formule explicite, est deux fois trop pessimiste sur cette plage.

**Ce que cette mesure ne dit pas.** Elle confirme un énoncé numérique du guide sur 2 000 points. C'est ce que l'on observerait si l'hypothèse de Riemann était vraie ; c'est aussi ce que l'on observerait avec un zéro un peu éloigné de la ligne, pour peu que son ordonnée soit grande. La sensibilité de la mesure se calcule, et elle se calcule mieux qu'elle ne se discute : B.5 s'en charge.

### Exercice 2 — L'erreur de $\psi$ hors des puissances de dix

Le tableau de B.1 ne montre que les puissances de dix. Calculer $\psi(x) - x$, puis son rapport à $\sqrt{x}$, pour $x = 5 \times 10^6$. La fonction `psi` de B.1 s'applique telle quelle.

In [10]:
# Exercice 2 - a completer : erreur de la fonction de compte en x = 5 x 10^6.
# Etape 1 : calculez psi(x_exo) - x_exo, avec la fonction psi definie en B.1.
# Etape 2 : divisez le resultat par sqrt(x_exo).
# Indice : les deux nombres demandes sont ceux des deux dernieres colonnes du tableau de B.1.
x_exo = 5 * 10 ** 6
ecart_exo = None    # TODO etudiant : psi(x_exo) - x_exo
rapport_exo = None  # TODO etudiant : ecart_exo / sqrt(x_exo)
print('Exercice a completer : psi(x) - x =', ecart_exo, '; rapport a sqrt(x) =', rapport_exo)

Exercice a completer : psi(x) - x = None ; rapport a sqrt(x) = None


### B.3 — Les zéros de $\zeta$, par la fonction $Z$ de Hardy

La formule explicite fait intervenir les zéros non triviaux $\rho = \beta + i\gamma$ de $\zeta$. Le carnet [12-zeros-simples-ligne-critique](12-zeros-simples-ligne-critique.ipynb) traite de leur simplicité et de leur position sur la ligne critique ; on n'y revient pas. Ici, il faut seulement **obtenir des zéros** avec les trois outils de B.0.

L'outil s'appelle la fonction $Z$ de Hardy :
$$Z(t) = e^{i\theta(t)}\,\zeta\!\left(\tfrac12 + it\right), \qquad \theta(t) = \Im\log\Gamma\!\left(\tfrac14 + \tfrac{it}{2}\right) - \tfrac{t}{2}\log\pi.$$
Le facteur $e^{i\theta(t)}$ est choisi pour rendre $Z$ **réelle** (*Cité* : construction classique, non démontrée ici). Trois propriétés la rendent utile :

- sur la ligne critique, $\zeta\!\left(\tfrac12 + it\right) = 0$ si et seulement si $Z(t) = 0$ ;
- $|Z(t)| = \left|\zeta\!\left(\tfrac12 + it\right)\right|$, donc $Z$ change de signe en chaque **zéro simple** de la ligne ;
- un zéro d'ordre pair ne produit **pas** de changement de signe : la méthode est aveugle à ce cas.

Le calcul de $\theta(t)$ utilise la formule de Stirling **avec décalage**. Stirling n'est précise que pour $|w|$ grand : on l'applique donc en $w = z + K$, puis on redescend par
$$\log\Gamma(z) = \log\Gamma(z + K) - \sum_{j=0}^{K-1} \log(z + j).$$
Les nombres de Bernoulli de B.0 donnent les termes de Stirling. Les zéros sont ensuite cherchés **par changements de signe** de $Z$, avec un pas de 0,1 puis une dichotomie.

**H5 — Hypothèse pré-enregistrée.** *Les deux extrémités de la liste calculée coïncideront avec celles du carnet 12 : $\gamma_1 = 14{,}134725$ à $10^{-6}$ près et $\gamma_{100} = 236{,}52423$ à $10^{-5}$ près. Et la partie imaginaire de $e^{i\theta(14)}\zeta\!\left(\tfrac12 + 14i\right)$ sera inférieure à $10^{-9}$ en valeur absolue.*

**Statut de la référence.** *Rapporté.* Les deux valeurs viennent de la sortie du carnet 12, qui les obtient par un autre chemin (la fonction `zetazero` de `mpmath`). Le présent carnet n'emploie que la bibliothèque standard : la comparaison est une **vérification croisée entre deux méthodes indépendantes**, pas une preuve. Elle ne dit rien des zéros au-delà du centième, ni des multiplicités.

In [11]:
def theta(t):
    """theta(t) = arg Gamma(1/4 + i t / 2) - (t/2) log pi, par la formule de Stirling avec decalage."""
    z = 0.25 + 0.5j * t
    K = int(t / 2) + 2                           # on remonte a w = z + K, ou Stirling est precis
    w = z + K
    lg = (w - 0.5) * cmath.log(w) - w + 0.5 * math.log(2 * math.pi)
    for m in range(1, 8):                        # termes de Stirling en 1 / w^(2m - 1)
        lg += float(B[2 * m]) / (2 * m * (2 * m - 1)) * w ** (1 - 2 * m)
    for j in range(K):                           # puis on redescend : log Gamma(w) - sum log(z + j)
        lg -= cmath.log(z + j)
    return lg.imag - 0.5 * t * math.log(math.pi)


def grand_z(t):
    """Z(t) = e^{i theta(t)} zeta(1/2 + i t) : la fonction de Hardy, reelle."""
    # Euler-Maclaurin a environ t termes sommes : suffisant ici, t restant petit devant 300.
    return (cmath.exp(1j * theta(t)) * hurwitz(0.5 + 1j * t, 1.0, max(10, int(t) + 1), 9, B)).real


def zeros(nb):
    """Les premiers nb zeros 1/2 + i gamma de zeta, par changements de signe de Z."""
    t, trouve = 10.0, []
    while len(trouve) < nb:
        pas = 0.1
        while grand_z(t) * grand_z(t + pas) > 0:     # avancer jusqu'au prochain changement de signe
            t += pas
        a, b = t, t + pas
        for _ in range(60):                      # dichotomie : 60 pas donnent la precision machine
            c = (a + b) / 2
            if grand_z(a) * grand_z(c) <= 0:
                b = c
            else:
                a = c
        trouve.append((a + b) / 2)
        t = b
    return trouve


cent_zeros = zeros(100)                          # cent zeros, pour la formule explicite de B.4

# H5a : le facteur e^{i theta} rend bien Z reelle (partie imaginaire nulle) ;
# H5b : premiere et centieme ordonnees egales aux valeurs du carnet 12, aux chiffres affiches.
h5a = abs((cmath.exp(1j * theta(14.0)) * hurwitz(0.5 + 14.0j, 1.0, 15, 9, B)).imag) < 1e-9
h5b = (f'{cent_zeros[0]:.6f}', f'{cent_zeros[99]:.5f}') == ('14.134725', '236.52423')
print(f'gamma_1 = {cent_zeros[0]}, gamma_2 = {cent_zeros[1]}, gamma_10 = {cent_zeros[9]}, gamma_100 = {cent_zeros[99]}')
print('Z(14) =', grand_z(14.0), '; Z(14.5) =', grand_z(14.5))
print('H5 :', 'CONFIRMEE' if (h5a and h5b) else 'REFUTEE')

gamma_1 = 14.134725141734695, gamma_2 = 21.022039638771552, gamma_10 = 49.773832477672286, gamma_100 = 236.52422966581616
Z(14) = -0.10562626777988432 ; Z(14.5) = 0.29735094506724286
H5 : CONFIRMEE


### Lecture du résultat

**H5 : CONFIRMÉE.** La liste calculée commence à $\gamma_1 = 14{,}1347251$ et son centième terme vaut $\gamma_{100} = 236{,}5242297$ : ce sont, aux chiffres affichés, les valeurs du carnet 12. La partie imaginaire testée est inférieure à $10^{-9}$ (*Mesuré*). Deux méthodes sans ligne de code en commun — `mpmath.zetazero` d'un côté, Euler–Maclaurin et la fonction $Z$ de l'autre — retrouvent les mêmes chiffres : c'est une vérification croisée, et elle ne vaut que pour les chiffres comparés.

Le second affichage montre le mécanisme : $Z(14) = -0{,}106$ et $Z(14{,}5) = +0{,}297$. La fonction est négative juste avant le premier zéro, positive juste après : elle a bien changé de signe entre les deux, à l'endroit où la dichotomie trouve $14{,}1347$.

**Portée.** Cent zéros, c'est très peu. C'est assez pour la formule explicite tronquée de B.4 ; ce n'est rien pour une conclusion sur l'hypothèse de Riemann, et la méthode ne verrait pas un zéro d'ordre pair.

### Exercice 3 — Compter des changements de signe

La fonction `zeros` de B.3 avance par pas de 0,1. **Combien de changements de signe de $Z$ y a-t-il entre $t = 20$ et $t = 30$ ?** On utilisera `grand_z` et une boucle, sans appeler `zeros`. On pourra comparer au nombre de zéros de la liste `cent_zeros` situés dans le même intervalle.

In [12]:
# Exercice 3 - a completer : changements de signe de Z entre t = 20 et t = 30.
# Etape 1 : echantillonnez grand_z(t) sur une grille de pas 0.1, de 20 a 30.
# Etape 2 : comptez les endroits ou deux valeurs successives sont de signes contraires.
# Indice : comparez votre resultat au nombre d'elements g de cent_zeros tels que 20 <= g <= 30.
changements_exo = None  # TODO etudiant : nombre de changements de signe de Z entre 20 et 30
print('Exercice a completer : nombre de changements de signe de Z entre 20 et 30 =', changements_exo)

Exercice a completer : nombre de changements de signe de Z entre 20 et 30 = None


### B.4 — La formule explicite, tronquée à cent zéros

La formule explicite relie la somme de von Mangoldt aux zéros de $\zeta$. Le guide l'écrit ainsi (§ 2) :

> « Read backwards, this is the explicit formula G(y) = log 2π − Σ_ρ y^ρ/(ρ(ρ−1)) + O(y^{−2}), in which ρ runs over the zeros of ζ in the critical strip, with multiplicity. »

Lue à l'envers, elle dit que $G(y) - \log 2\pi$ est **moins** la somme des termes $y^\rho\big/\big(\rho(\rho-1)\big)$, pris sur tous les zéros non triviaux. Un zéro $\rho = \beta + i\gamma$ apporte un terme de module $y^\beta\big/\big|\rho(\rho-1)\big|$.

- **Sous l'hypothèse de Riemann**, $\beta = 1/2$ pour tous les zéros. Chaque terme est alors de taille $\sqrt{y}\big/\big|\rho(\rho-1)\big|$, et la somme de ces tailles vaut $\sqrt{y}\,(2 + \gamma - \log 4\pi)$ : la constante 0,04619 calculée en B.0. D'où l'inégalité que le guide en tire : $|G(y) - \log 2\pi| \le 0{,}0462\,\sqrt{y} + O(y^{-2})$.
- **Sans elle**, un zéro de partie réelle $\beta > 1/2$ apporte un terme en $y^\beta$, qui l'emporte sur $\sqrt{y}$ quand $y$ grandit.

Le statut de la formule dans ce carnet est *Cité* : elle est classique, citée depuis le guide, et non démontrée ici.

**Ce que le calcul teste.** Si la formule est exacte, alors $G(y) - \log 2\pi$ **plus** la somme des termes des zéros doit s'annuler, à l'erreur $O(y^{-2})$ près. On ne dispose que de cent zéros : les termes manquants sont de l'ordre de $\sqrt{y}/\gamma^2$ avec $\gamma > 236$, donc petits devant $\sqrt{y}$. Le résidu attendu n'est pas nul, mais **petit une fois divisé par $\sqrt{y}$**. On le mesure en trois points, et dans une fenêtre de neuf points autour de chacun, parce qu'un point isolé peut tomber sur une phase favorable.

**H6 — Hypothèse pré-enregistrée.** *En chacun des points $y = 100$, $10^4$ et $10^6$, et sur la fenêtre de neuf points répartis à ±8 % autour de chacun, le résidu divisé par $\sqrt{y}$ sera inférieur à 0,01.*

**Ce que ce test ne montre pas.** Les zéros sont placés en $\frac12 + i\gamma$ parce que c'est là que B.3 les a cherchés. Le test vérifie la **formule**, pas l'**hypothèse**.

In [13]:
def somme_zeros(y, liste):
    """Somme des y^rho / (rho (rho - 1)), chaque zero pris avec son conjugue."""
    s = 0j
    for g in liste:
        rho = 0.5 + 1j * g                           # zero sur la ligne critique, d ordonnee g
        s += cmath.exp(rho * cmath.log(y)) / (rho * (rho - 1))
    return 2 * s.real                               # le conjugue apporte la meme partie reelle


# Residu de la formule explicite : (G - log 2 pi) + somme des termes de zeros.
# Il doit etre petit devant sqrt(y) ; sa principale source est le tronquage a cent zeros.
print('     y      residu : (G - log 2 pi) + somme sur 100 zeros     |residu| / sqrt(y)')
h6 = True
for y in (100.0, 10 ** 4, 10.0 ** 6):
    r = (grand_g(y) - LOG2PI) + somme_zeros(y, cent_zeros)
    fenetre = [y * 1.02 ** d for d in range(-4, 5)]          # 9 points autour de y, a 8 pour cent
    pire = max(abs((grand_g(v) - LOG2PI) + somme_zeros(v, cent_zeros)) / sqrt(v) for v in fenetre)
    h6 = h6 and pire < 0.01                                  # critere porte sur le pire point de la fenetre
    print(f'{y:>8.0f}   {r:+.5f}   {abs(r) / sqrt(y):.2e}   pire de la fenetre : {pire:.2e}')
print('H6 :', 'CONFIRMEE' if h6 else 'REFUTEE')

     y      residu : (G - log 2 pi) + somme sur 100 zeros     |residu| / sqrt(y)
     100   +0.00301   3.01e-04   pire de la fenetre : 3.01e-04
   10000   +0.03266   3.27e-04   pire de la fenetre : 3.27e-04
 1000000   -0.00472   4.72e-06   pire de la fenetre : 3.98e-04
H6 : CONFIRMEE


### Lecture du résultat

**H6 : CONFIRMÉE.** Le résidu vaut $+0{,}00301$ en $y = 100$, $+0{,}03266$ en $10^4$ et $-0{,}00472$ en $10^6$ (*Mesuré*). Ramené à $\sqrt{y}$, il est de l'ordre de $3 \times 10^{-4}$ aux deux premiers points et de $5 \times 10^{-6}$ au troisième ; le pire point de chaque fenêtre reste en dessous de $4 \times 10^{-4}$, soit vingt-cinq fois moins que le seuil de 0,01.

À $y = 100$, l'écart $G - \log 2\pi$ vaut $-0{,}116$ (B.2). Les cent premiers zéros en expliquent plus de 97 % ; d'après la formule, le résidu est la part des zéros manquants.

**Lecture honnête.** La formule explicite, tronquée à cent zéros tous pris sur la ligne critique, reconstitue $G$ à moins de quelques dix-millièmes de $\sqrt{y}$. C'est une vérification numérique d'une identité classique, sur une plage bornée. Elle ne dit rien de l'hypothèse de Riemann : elle montre qu'avec des zéros sur la ligne, les comptes sont justes. Un zéro hors de la ligne ajouterait son propre terme, et c'est ce que la section suivante met en évidence.

### B.5 — Un zéro hors de la ligne se verrait-il ?

Deux questions séparent l'expérience de B.2 de ce dont une preuve a besoin. Toutes deux sont *Citées* : le texte qui suit rapporte le guide (§ 2 et § 3), sans le démontrer.

**Une borne supérieure, sur des fenêtres.** Après avoir écrit l'inégalité $|G(y) - \log 2\pi| \le 0{,}0462\sqrt{y}$ sous l'hypothèse de Riemann, le guide ajoute :

> « The proof needs far less: an upper bound, on windows. »

> « Landau's theorem says that the Mellin transform of a function of constant sign has a singularity at the real point of its abscissa of convergence. Applied to Cyθ−G(y) it shows that a bound G(y) ≤ Cyθ for all large y would exclude zeros of ζ with Re s > θ. The arithmetic of §4–6 yields such a bound only on certain intervals, and only for several characters together. So the theorem has to be localized, and the zeros of different L-functions must be kept from cancelling. »

Trois choses à retenir. Une **borne supérieure** $G(y) \le Cy^\theta$, valable pour tout $y$ assez grand, suffirait à exclure les zéros de partie réelle supérieure à $\theta$ : c'est l'idée du théorème de Landau. Mais l'arithmétique de la preuve ne l'obtient que sur certains intervalles, et pour plusieurs caractères à la fois. Le guide définit donc une **famille positive** $F$ : un ensemble fini de caractères de Dirichlet, avec des poids $c_\chi > 0$, et la somme
$$G_F(y) = \sum_{\chi \in F} c_\chi\,\mathrm{Re}\,g_\chi(y), \qquad g_1 := G \text{ pour le caractère trivial.}$$
Une **fenêtre** est un intervalle $[Y, Y^{1+\delta}]$, avec $\delta > 0$. La proposition du guide dit alors ceci :

> « Proposition. Let F be a positive family, 0 < θ < 1 and δ > 0. Suppose that some L(s, χ) with χ∈F has a zero with Re s > θ. Then for every real number C there is Y0 such that every window [Y, Y1+δ] with Y≥Y0 contains a point y with GF(y) > Cyθ. »

> « Thus a zero shows itself, with a definite sign, in every late window. »

Un zéro dépasse la borne **dans toute fenêtre assez tardive**, avec un signe fixé. C'est plus exigeant, et plus commode, qu'un dépassement quelque part.

**La longueur $\sqrt{X}$ porte un exposant $1/2 + 9\varepsilon$.** Pour un caractère $\psi$, le guide forme la famille $P(\psi)$ des caractères qui induisent les puissances de $\psi$, avec le poids 1 sur le caractère trivial et $1/(3^o + 1)$ sur chacun des autres, et pose $G_\psi = G_{P(\psi)}$. La moitié arithmétique de la preuve s'énonce ainsi :

> « Statement (⋆). There is ε1 > 0 such that for every 0 < ε < ε1 there are arbitrarily large Y with Gψ(x) ≤ x1/2+9ε for all x∈[Y, Y3/2]. »

Le 9 de l'exposant est un compte de pertes, que la partie C retrouvera. Et l'ordre des choix importe :

> « So θ is chosen after the zero and ε after θ. No bound at ε = 0 and no uniformity in ε is needed, and the windows may depend on everything. »

La contradiction vient alors de ce que, pour un zéro $\rho$, on prend $\theta$ entre $1/2$ et $\mathrm{Re}\,\rho$, puis $\varepsilon$ tel que $1/2 + 9\varepsilon \le \theta$ : les fenêtres tardives de (⋆) contredisent la proposition.

**Expérience.** On ne dispose pas de la preuve arithmétique, et on n'en reproduit rien. On peut en revanche **voir le mécanisme** sur une fonction construite pour cela : $G$ augmentée du terme qu'apporterait un zéro supplémentaire de partie réelle 0,9 et d'ordonnée $\gamma_1$. Ce n'est pas une somme de von Mangoldt. Avec $\theta = 0{,}6$, situé entre $1/2$ et 0,9, on mesure sur les fenêtres $[Y, Y^{3/2}]$, pour $Y = 10^2$, $10^3$, $10^4$, le maximum de $\dfrac{|G - \log 2\pi|}{y^{0{,}6}}$, pour la vraie somme et pour la fonction construite.

**H7 — Hypothèse pré-enregistrée.** *Pour la vraie somme, le maximum de la fenêtre $Y = 10^4$ sera inférieur à celui de la fenêtre $Y = 10^2$. Pour la fonction construite, il sera plus de trois fois celui de la fenêtre $Y = 10^2$.*

**Sensibilité de la mesure de B.2.** Le même terme répond à la question laissée ouverte en B.2. Un zéro en $\beta + i\gamma$ apporte environ $2y^\beta/\gamma^2$ à $G - \log 2\pi$, son conjugué compris. Il atteint le seuil $0{,}023\sqrt{y}$ dès que $\gamma^2 < 2y^{\beta - 1/2}\big/0{,}023$. Le calcul donne, à $y = 10^7$, l'ordonnée limite pour trois valeurs de $\beta$.

In [14]:
def faux_temoin(y, beta, gamma1):
    """G(y) augmentee du terme d'un zero deplace a beta + i gamma_1, pris avec son conjugue."""
    rho = beta + 1j * gamma1
    return grand_g(y) + 2 * (cmath.exp(rho * cmath.log(y)) / (rho * (rho - 1))).real


# Meme maximum relatif sur la fenetre [Y, Y^(3/2)] : pour la vraie somme, puis pour la fonction construite.
exposant = 0.6                                  # theta, entre 1/2 et la partie reelle 0.9 du zero fictif
max_vrai, max_faux = [], []
print('     Y     max |G - log 2 pi| / y^0.6    idem, zero deplace a 0.9 + i gamma_1')
for Y in (100.0, 10 ** 3, 10.0 ** 4):
    fenetre = [Y * (Y ** 0.5) ** (j / 30) for j in range(31)]    # 31 points sur [Y, Y^(3/2)]
    max_vrai.append(max(abs(grand_g(y) - LOG2PI) / y ** exposant for y in fenetre))
    max_faux.append(max(abs(faux_temoin(y, 0.9, cent_zeros[0]) - LOG2PI) / y ** exposant for y in fenetre))
    print(f'{Y:>7.0f}   {max_vrai[-1]:.4f}   {max_faux[-1]:.4f}')
print('facteur de croissance entre Y = 10^2 et Y = 10^4 : vrai', round(max_vrai[2] / max_vrai[0], 2),
      '; faux', round(max_faux[2] / max_faux[0], 2))
print('facteur attendu pour un zero de partie reelle 0.9 :', round(((10 ** 4) ** 1.5 / (100.0) ** 1.5) ** (0.9 - exposant), 2))
h7 = max_vrai[2] < max_vrai[0] and max_faux[2] / max_faux[0] > 3
print('H7 :', 'CONFIRMEE' if h7 else 'REFUTEE')

# Sensibilite de la mesure de B.2 : un zero en beta + i gamma (son conjugue compris) apporte a
# G - log 2 pi un terme de taille proche de 2 y^beta / gamma^2. Il atteint le seuil 0.023 sqrt(y)
# des que gamma^2 < 2 y^(beta - 1/2) / 0.023.
print('a y = 10^7, ordonnee en dessous de laquelle le terme d un zero de partie reelle beta atteint 0.023 sqrt(y) :')
for beta in (0.5001, 0.6, 0.9):
    print(f'   beta = {beta} : gamma < {sqrt(2 * (10.0 ** 7) ** (beta - 0.5) / 0.023):.1f}')

     Y     max |G - log 2 pi| / y^0.6    idem, zero deplace a 0.9 + i gamma_1
    100   0.0108   0.0708
   1000   0.0069   0.1988
  10000   0.0056   0.5330
facteur de croissance entre Y = 10^2 et Y = 10^4 : vrai 0.52 ; faux 7.53
facteur attendu pour un zero de partie reelle 0.9 : 7.94
H7 : CONFIRMEE
a y = 10^7, ordonnee en dessous de laquelle le terme d un zero de partie reelle beta atteint 0.023 sqrt(y) :
   beta = 0.5001 : gamma < 9.3
   beta = 0.6 : gamma < 20.9
   beta = 0.9 : gamma < 234.2


### Lecture du résultat

**H7 : CONFIRMÉE** (*Mesuré*). Pour la vraie somme, le maximum relatif **diminue** d'une fenêtre à l'autre : 0,0108, puis 0,0069, puis 0,0056. Une borne $G(y) \le Cy^{0{,}6}$, avec le même $C$ sur les trois fenêtres, est donc compatible avec la mesure. Pour la fonction construite, il **monte** : 0,0708, puis 0,1988, puis 0,5330, d'un facteur 7,53 entre les fenêtres extrêmes.

Le compte est bon. Le terme d'un zéro de partie réelle 0,9 croît comme $y^{0,9}$ ; divisé par $y^{0{,}6}$, il croît comme $y^{0{,}3}$. Entre les extrémités des deux fenêtres, $10^3$ et $10^6$, le facteur attendu est $(10^6/10^3)^{0{,}3} = 7{,}94$. L'écart avec 7,53 vient de la phase du terme et de la vraie somme qui s'y ajoute.

C'est le mécanisme de la proposition du guide, vu de loin : un zéro dépasse la borne dans toute fenêtre assez tardive. L'expérience n'en montre que la **croissance**. Le signe, lui, n'est pas fixé ici, puisque la valeur absolue est prise et que le zéro fictif s'ajoute à la vraie somme.

**Ce que l'expérience n'est pas.** Elle ne construit aucune somme $G_F$ : ni famille positive, ni caractères, ni localisation du théorème de Landau. Elle ajoute à une fonction calculée le terme d'un zéro qui n'existe pas. Elle fait seulement voir pourquoi une borne d'exposant $1/2 + 9\varepsilon$ peut suffire : ce qui compte est que cet exposant reste inférieur à la partie réelle du zéro que l'on cherche à exclure.

**Sensibilité de la mesure de B.2.** Les trois dernières lignes donnent, à $y = 10^7$, l'ordonnée en dessous de laquelle le terme d'un zéro atteint le seuil de 0,023 : 9,3 pour $\beta = 0{,}5001$, 20,9 pour $\beta = 0{,}6$, 234,2 pour $\beta = 0{,}9$ (en remplaçant $\big|\rho(\rho-1)\big|$ par $\gamma^2$). Un zéro de partie réelle 0,5001 ne se verrait donc que s'il était très bas ; un zéro de partie réelle 0,6, que s'il avait une ordonnée inférieure à 21. Autrement dit, B.2 établit peu de chose sur les zéros : elle dit qu'aucun zéro bas et éloigné de la ligne ne vient troubler $G$ entre 10 et $10^7$. C'est une raison pour laquelle une preuve ne se contente pas d'une mesure : elle a besoin de la borne sur des fenêtres arbitrairement tardives.

## Partie C — La longueur $\sqrt{X}$ en échelle finie

La partie A a calculé une brique de l'argument de Stepanov, la partie B les sommes de von Mangoldt. Reste la troisième brique demandée par l'issue : **la longueur $\sqrt{X}$, en échelle finie**. Ici encore, la chaîne automorphe est **citée, jamais reproduite**.

**Ce que l'auteur annonce** (*Annoncé*, note du 20 novembre 2026) :

> « In Stepanov's argument resolving Riemann over function fields one uses a divisor involving multiplicity. This argument is a descendant of Stepanov's, but multiplicity is hard to find over the integers, corresponding to something like differentiating whole numbers. »

> « for the von Mangoldt sum of length X, [·]'s suggested length of chain of theta lifts, in view of each mimicking multiplicity, was √X. (And yet, what else could it be?) »

Le crochet $[\cdot]$ masque un nom : la surface publique de ce dépôt ne porte aucun nom d'entreprise ni de produit. L'entrée 8 de la chronologie de la note dit la même chose autrement :

> « Iterated theta lifts U(X₀) → U(X₁) → ⋯ → U(X_m) between totally definite unitary groups, π ↦→ π ⊞ μ_j[t], wins multiplicity m. Time to take m like √X... »

Sur l'état du texte, la note est explicite :

> « The argument is incredibly long at the moment »

**Ce que le guide en dit** (*Cité*, § 4). Trois idées :

1. **Le prix de la divisibilité.** « The announcement records (entry 2) that for every prime p inert in K and every integer a ≥ 1 the number D^a θ(0)/Ω^a is divisible by p^(a/p + a/p³ + ⋯ − O(log a)), much as a! is divisible by p^(⌊a/p⌋ + ⌊a/p²⌋ + ⋯). Now a non-zero algebraic integer that is divisible by many primes cannot be small: by the product formula, the sum of log p times these exponents, over the inert primes, is at most the logarithm of its size at the infinite places. »
2. **Une tour cyclotomique** : « This cyclotomic tower plays the part of the finite extensions of 𝔽 in Stepanov's proof. »
3. **Un compromis** : « A short chain gains too little and a long one costs too much: the optimum is m ≍ √x. In the archive √x δ₁ ≤ m ≤ 2√x δ₁ with δ₁ = x^(−ε/8). »

Le guide prévient lui-même de la portée de ces lectures : « §§4–7 are read here from the Lean source and from the announcement, not re-derived. »

**Ce que le carnet calcule.** Les trois sections qui suivent prennent ces citations une à une et calculent ce qui se calcule avec de l'arithmétique entière et les outils de la partie B :

- **C.1** : la divisibilité de $a!$ par les puissances des nombres premiers, prise pour modèle par le guide ;
- **C.2** : les deux membres $\mathrm{ASK}$ et $\mathrm{MAIN}$ du guide, sur un cas d'école ;
- **C.3** : le compromis entre une chaîne trop courte et une chaîne trop longue.

### C.1 — Une factorielle et ses valuations

Soit $v_p(n)$ l'exposant de $p$ dans la décomposition de $n$ en facteurs premiers. **La formule de Legendre** donne celui de $a!$ :
$$v_p(a!) = \left\lfloor \tfrac{a}{p} \right\rfloor + \left\lfloor \tfrac{a}{p^2} \right\rfloor + \left\lfloor \tfrac{a}{p^3} \right\rfloor + \cdots$$
Elle se démontre en une ligne : parmi les entiers de 1 à $a$, exactement $\lfloor a/p^j \rfloor$ sont divisibles par $p^j$, et chacun apporte autant de facteurs $p$.

En la combinant avec l'écriture de $a$ en base $p$, on obtient une identité remarquable. Si $s_p(a)$ désigne la somme des chiffres de $a$ en base $p$ :
$$(p - 1)\,v_p(a!) = a - s_p(a).$$
La preuve tient en deux lignes. On écrit $a = \sum_i d_i\,p^i$, avec $0 \le d_i < p$, d'où $s_p(a) = \sum_i d_i$ et $\lfloor a/p^j \rfloor = \sum_{i \ge j} d_i\,p^{i-j}$. En sommant $(p-1)\,\lfloor a/p^j \rfloor$ sur $j \ge 1$, chaque chiffre $d_i$ apporte $d_i\,(p - 1)\,(1 + p + \cdots + p^{i-1}) = d_i\,(p^i - 1)$ ; la somme vaut donc $a - s_p(a)$.

**H8 — Hypothèse pré-enregistrée.** *L'identité $(p - 1)\,v_p(a!) = a - s_p(a)$ n'aura aucune exception sur $1 \le a \le 10^4$, pour les onze premiers nombres premiers (de 2 à 31). Pour $a = 50$, 200 et 1000, le produit des $p^{v_p(a!)}$, pris sur les nombres premiers $p \le a$, sera exactement égal à $a!$. Et 1000! finira par 249 zéros.*

La formule de Legendre est classique et démontrée ci-dessus (*Cité*, preuve incluse). Les trois vérifications sont *Mesurées* : elles ne surprennent pas, et c'est le but. Elles établissent l'outil avant de lire ce que le guide en fait.

In [15]:
def valuation(p, a):
    """v_p(a!) par la formule de Legendre : a // p + a // p^2 + ..."""
    v, q = 0, p
    while q <= a:                                # les termes s'annulent des que p^j depasse a
        v += a // q
        q *= p
    return v


def somme_chiffres(p, a):
    """Somme des chiffres de a ecrit en base p."""
    s = 0
    while a:
        s += a % p                               # chiffre des unites, puis on le retire
        a //= p
    return s


# Nombres premiers <= 1000, par division essayee.
premiers_c = [n for n in range(2, 1001) if all(n % d for d in range(2, int(sqrt(n)) + 1))]

# H8, partie 1 : l'identite (p - 1) v_p(a!) = a - s_p(a) n'a aucune exception.
ecarts = sum(valuation(p, a) * (p - 1) != a - somme_chiffres(p, a)
             for a in range(1, 10 ** 4 + 1) for p in premiers_c[:11])
h8 = ecarts == 0
print('v_p(a!) (p - 1) = a - s_p(a) sur a <= 10^4 et les onze premiers <= 31 : ecarts =', ecarts)

# H8, partie 2 : a! se reconstruit-il a partir de ses valuations ?
for a in (50, 200, 1000):
    prod = 1
    for p in premiers_c:
        if p <= a:                               # seuls les premiers <= a divisent a!
            prod *= p ** valuation(p, a)
    h8 = h8 and prod == factorial(a)
    print(f'a = {a:>4} : le produit des p^v_p(a!) est egal a a! : {prod == factorial(a)}')

# H8, partie 3 : les zeros finals de 1000!, en base dix.
zeros_finaux = min(valuation(5, 1000), valuation(2, 1000))   # un 2 et un 5 font un 10
print('v_5(1000!) =', valuation(5, 1000), 'et v_2(1000!) =', valuation(2, 1000),
      ': 1000! finit par', zeros_finaux, 'zeros')
h8 = h8 and zeros_finaux == 249                  # valeur pre-enregistree
print('H8 :', 'CONFIRMEE' if h8 else 'REFUTEE')

v_p(a!) (p - 1) = a - s_p(a) sur a <= 10^4 et les onze premiers <= 31 : ecarts =

 0
a =   50 : le produit des p^v_p(a!) est egal a a! : True
a =  200 : le produit des p^v_p(a!) est egal a a! : True
a = 1000 : le produit des p^v_p(a!) est egal a a! : True
v_5(1000!) = 249 et v_2(1000!) = 994 : 1000! finit par 249 zeros
H8 : CONFIRMEE


### Lecture du résultat

**H8 : CONFIRMÉE** (*Mesuré*). L'identité $(p-1)\,v_p(a!) = a - s_p(a)$ n'a aucune exception sur les 110 000 couples testés. Les trois factorielles se reconstruisent **exactement** à partir de leurs valuations : $a! = \prod_{p \le a} p^{v_p(a!)}$. Et $v_5(1000!) = 249$, $v_2(1000!) = 994$ : 1000! finit donc par 249 zéros, un 2 et un 5 faisant un 10.

**Ce que cela établit, et ce que cela n'établit pas.** La divisibilité d'un entier par les puissances des nombres premiers est une information **exacte**, qui se calcule sans aucune approximation. C'est le modèle que prend le guide quand il écrit qu'un entier algébrique non nul, divisible par beaucoup de nombres premiers, « cannot be small ». Ici, seul le modèle a été vérifié, sur des factorielles. L'énoncé du guide porte sur un autre objet, $D^a\theta(0)/\Omega^a$, et il n'est **pas reproduit** : ni sa preuve, ni l'exposant $a/p + a/p^3 + \cdots$, qui n'est pas celui de $v_p(a!)$.

### Exercice 4 — Les zéros finals d'une autre factorielle

**Combien 2019! finit-il par de zéros, en écriture décimale ?** On utilisera `valuation`, et rien d'autre. La comparaison entre $v_2$ et $v_5$ dans l'exemple de C.1 donne la règle à appliquer.

In [16]:
# Exercice 4 - a completer : nombre de zeros finals de 2019!, en ecriture decimale.
# Etape 1 : calculez valuation(5, 2019) et valuation(2, 2019).
# Etape 2 : prenez le plus petit des deux : en base dix, un 2 et un 5 font un 10.
# Indice : la derniere ligne de C.1 fait ce calcul pour 1000!.
zeros_exo = None  # TODO etudiant : nombre de zeros finals de 2019!
print('Exercice a completer : nombre de zeros finals de 2019! =', zeros_exo)

Exercice a completer : nombre de zeros finals de 2019! = None


### C.2 — Premiers inertes, caractère de Dirichlet, $\mathrm{ASK}$ contre $\mathrm{MAIN}$

Le guide (§ 4) introduit deux nombres premiers auxiliaires $q$ et $d$, soumis à des conditions données en § 5, et pose $K = \mathbb{Q}(\sqrt{-d})$. Le **caractère de Dirichlet quadratique** de $K$ vaut, sur les nombres premiers :

> « χ_{−d}(p) = 1 if the prime p splits in K, −1 if it is inert, and 0 if p = d. »

Le membre de gauche de l'inégalité s'appelle $\mathrm{ASK}$, et fait intervenir la rampe $(u)^+ := \max(u, 0)$ de la partie B :
$$\mathrm{ASK}(a) = \sum_{n \ge 1} \Lambda(n)\,\big(1 - \chi_{-d}(n)\big)\left(\tfrac{a}{n} - w_n\right)^+.$$
Les poids $w_n$ y sont proches d'une valeur commune $b := e^{x^\varepsilon}$, et les hauteurs $a$ vérifient $a/b \approx x$. Le guide décrit le facteur $1 - \chi_{-d}(n)$ ainsi :

> « The factor 1 −χ−d(𝑛) equals 2 at the odd powers of primes inert in K, 1 at the powers of d, and 0 at all other prime powers. »

Le membre principal, lui, ne dépend que de $a$, de $w$ et de la valeur en 1 de la fonction $L$ de $\chi_{-d}$ :
$$\mathrm{MAIN}(a) = a\left(\log a - 1 - \gamma + \frac{L'}{L}(1, \chi_{-d}) - \log w\right).$$

Enfin le guide énonce l'identité qui relie tout cela à la partie B. Si tous les poids sont égaux à un même nombre $w$ et si $y = a/w$ :

> « The definitions of §2 then give at once ASK(a) − MAIN(a) = w(G(y) − gχ−d(y)). »

Et il dit ce que l'inégalité donne alors :

> « So the outcome is an upper bound of size about √x, at y ≈ x, for a combination of the centred sums of §2. It is not yet (⋆): the combination involves χ−d instead of ψ, and with a minus sign. Unequal weights repair this, in §5. »

Tous ces énoncés sont *Cités*. Rien n'en est démontré ici.

**Avertissement : un cas d'école, pas le cadre de la preuve.** Le calcul qui suit prend $d = 3$, donc $K = \mathbb{Q}(\sqrt{-3})$, et $w = 1$. Or le $d$ du guide est un nombre premier auxiliaire soumis à des conditions de congruence et de taille, que l'on choisit **fenêtre par fenêtre**. La valeur $d = 3$ ne les remplit pas. Le calcul illustre donc la **forme** des deux membres — même rampe, même centrage par $L'/L(1,\chi)$ — dans une plage de paramètres qui n'est **pas** celle de la preuve. Avec $d = 3$, le caractère est simple :
$$\chi(n) = \begin{cases} 1 & \text{si } n \equiv 1 \pmod 3, \\ -1 & \text{si } n \equiv 2 \pmod 3, \\ 0 & \text{si } 3 \mid n.\end{cases}$$
Un nombre premier $p \ne 3$ est **inerte** si $p \equiv 2 \pmod 3$, **décomposé** si $p \equiv 1 \pmod 3$.

La valeur $L'/L(1, \chi)$ demande une précaution d'ordre numérique. $L(s, \chi)$ n'a **pas de pôle** en $s = 1$, puisque $\chi$ n'est pas principal. Mais le terme intégral de la formule d'Euler–Maclaurin, en B.0, divise par $s - 1$ : pour $\zeta$, c'est le pôle ; pour $L(s,\chi)$, les deux termes se compensent, et le programme ne sait pas faire cette compensation en $s = 1$ exactement. On tourne donc autour : on évalue $L$ en $1 \pm h$ avec $h = 10^{-4}$, et on approche la dérivée par une différence centrée, la valeur par leur moyenne.

**H9 — Hypothèse pré-enregistrée.** *Cinq affirmations seront vérifiées :*

1. *pour tout nombre premier $p < 200$ autre que 3, $p$ s'écrit $r^2 - rt + t^2$ si et seulement si $p \equiv 1 \pmod 3$ ;*
2. *sur toutes les puissances de nombres premiers jusqu'à $10^7$, le poids $1 - \chi(n)$ vaut 2 aux puissances impaires des nombres premiers inertes, 1 aux puissances de 3, et 0 ailleurs, sans un seul écart ;*
3. *$L(1, \chi)$ coïncidera avec la valeur classique $\pi / (3\sqrt{3})$ à $10^{-8}$ près ;*
4. *$|\mathrm{ASK}(a) - \mathrm{MAIN}(a)|$ sera inférieur à $\sqrt{a}$ pour $a = 10^2, 10^3, \ldots, 10^7$ ;*
5. *l'écart entre $\mathrm{ASK}(a) - \mathrm{MAIN}(a)$ et $G(a) - g_\chi(a)$, rapporté à $a$, restera inférieur à $10^{-12}$ aux mêmes points.*

*Les points 1 et 3 sont des théorèmes classiques, pris ici comme témoins de l'outil. Le point 2 reprend la phrase du guide citée plus haut. Les points 4 et 5 comparent le calcul aux définitions du guide.*

In [17]:
def chi(n):
    """Caractere quadratique de Q(sqrt(-3)) : 1 si n = 1 mod 3, -1 si n = 2 mod 3, 0 sinon."""
    r = n % 3
    return 1 if r == 1 else (-1 if r == 2 else 0)

def dirichlet(s):
    """L(s, chi) = somme des chi(n) / n^s, par la fonction zeta de Hurwitz. Jamais en s = 1."""
    # Les classes 1 et 2 modulo 3 portent le poids +1 et -1 : deux fonctions zeta de Hurwitz.
    return 3 ** (-s) * (hurwitz(s, 1 / 3, 30, 8, B) - hurwitz(s, 2 / 3, 30, 8, B))

# H9, partie 3. L(s, chi) est reguliere en s = 1 ; mais le terme integral d Euler-Maclaurin divise
# par s - 1. On evalue donc de part et d autre, puis on prend une difference centree pour la derivee
# et une moyenne pour la valeur.
h = 1e-4
derivee = ((dirichlet(1 + h) - dirichlet(1 - h)) / (2 * h)).real
valeur = ((dirichlet(1 + h) + dirichlet(1 - h)) / 2).real
quotient = derivee / valeur                      # L'(1, chi) / L(1, chi)
l1 = math.pi / (3 * sqrt(3))                     # valeur classique de L(1, chi)

# H9, partie 1. Les valeurs de r^2 - r t + t^2 pour 0 <= r, t <= 19 : bornes larges, car toute
# representation d un nombre premier p par cette forme a r^2 <= 4 p / 3, donc |r|, |t| <= 16 pour p < 200.
normes = {(r * r - r * t + t * t) for r in range(20) for t in range(20)}
representes = all(((p in normes) == (p % 3 == 1)) for p in premiers_c if p < 200 and p != 3)
print('p = r^2 - r t + t^2 pour p premier < 200 : exactement les p = 1 mod 3 :', representes)

def poids_annonce(p, e):
    """Poids 1 - chi(p^e) annonce par le guide : 2 si p est inerte et e impair, 1 si p = 3, 0 sinon."""
    return 2 if (p % 3 == 2 and e % 2 == 1) else (1 if p == 3 else 0)

# H9, partie 2. Sur chaque puissance de nombre premier n = p^e de B.1, le poids 1 - chi(n) calcule
# par le caractere doit etre celui que le guide annonce.
ecarts_poids = 0
for n, logp in puissances:
    p, e = round(math.exp(logp)), round(math.log(n) / logp)      # on retrouve p et e depuis les listes de B.1
    ecarts_poids += (p ** e != n) or ((1 - chi(n)) != poids_annonce(p, e))
print('poids 1 - chi(n) sur les', len(puissances), 'puissances de premiers : 2 pour', sum(chi(n) == -1 for n in indice),
      'puissances impaires de premiers inerties,', '1 pour', sum(chi(n) == 0 for n in indice),
      'puissances de 3, 0 sinon ; ecarts au poids annonce =', ecarts_poids)
print('L(1, chi) =', repr(valeur), '; pi / (3 sqrt 3) =', repr(l1))
print("L'(1, chi) =", derivee, "; L'(1, chi) / L(1, chi) =", quotient)

# Sommes cumulees des poids : demandee pour ASK, centree pour g_chi.
somme_demandee_sur_n = [0.0]                     # sommes partielles de Lambda(n) (1 - chi(n)) / n
somme_demandee = [0.0]                           # et de Lambda(n) (1 - chi(n))
somme_centree_sur_n = [0.0]                      # idem avec chi(n) seul : la somme centree g_chi
somme_centree = [0.0]
for n, logp in puissances:
    poids = logp * (1 - chi(n))
    somme_demandee_sur_n.append(somme_demandee_sur_n[-1] + poids / n)
    somme_demandee.append(somme_demandee[-1] + poids)
    poids_chi = logp * chi(n)
    somme_centree_sur_n.append(somme_centree_sur_n[-1] + poids_chi / n)
    somme_centree.append(somme_centree[-1] + poids_chi)

def ask(a):
    """ASK(a) = somme des Lambda(n) (1 - chi(n)) (a/n - 1)^+ sur n <= a."""
    # Pour n <= a, la rampe (a/n - 1)^+ vaut a/n - 1 : la somme est a fois la premiere liste, moins la seconde.
    i = bisect.bisect_right(indice, a)
    return a * somme_demandee_sur_n[i] - somme_demandee[i]

def main(a):
    """MAIN(a) = a (log a - 1 - gamma + L'(1, chi) / L(1, chi)), pour la valeur w = 1."""
    return a * (math.log(a) - 1 - gamma + quotient)

def g_chi(a):
    """g_chi(y) = somme des Lambda(n) chi(n) (y/n - 1) + y L'(1, chi) / L(1, chi)."""
    i = bisect.bisect_right(indice, a)
    return a * somme_centree_sur_n[i] - somme_centree[i] + a * quotient

print('      a       ASK(a) / a      MAIN(a) / a     (ASK - MAIN) / sqrt(a)   |ASK - MAIN - (G - g)(a)| / a')
h9 = representes and ecarts_poids == 0 and abs(valeur - l1) < 1e-8   # parties 1, 2 et 3
ecart_identite = 0.0                             # ASK - MAIN doit valoir G(a) - g(a) pour w = 1
for k in range(2, 8):
    a = 10.0 ** k
    h9 = h9 and abs(ask(a) - main(a)) < sqrt(a)                      # partie 4
    ecart_identite = max(ecart_identite, abs((ask(a) - main(a)) - (grand_g(a) - g_chi(a))) / a)
    print(f'{a:>8.0f}   {ask(a) / a:>+12.6f}   {main(a) / a:>+12.6f}   {(ask(a) - main(a)) / sqrt(a):+.5f}   {ecart_identite:.1e}')
h9 = h9 and ecart_identite < 1e-12               # partie 5 : la somme flottante porte l'arrondi cumule
print('H9 :', 'CONFIRMEE' if h9 else 'REFUTEE')

p = r^2 - r t + t^2 pour p premier < 200 : exactement les p = 1 mod 3 : True


poids 1 - chi(n) sur les 665134 puissances de premiers : 2 pour 332424 puissances impaires de premiers inerties, 1 pour 14 puissances de 3, 0 sinon ; ecarts au poids annonce = 0
L(1, chi) = 0.6045997875853761 ; pi / (3 sqrt 3) = 0.6045997880780726
L'(1, chi) = 0.2226629858498308 ; L'(1, chi) / L(1, chi) = 0.3682816144198336


      a       ASK(a) / a      MAIN(a) / a     (ASK - MAIN) / sqrt(a)   |ASK - MAIN - (G - g)(a)| / a
     100      +3.402652      +3.396236   +0.06416   5.0e-16
    1000      +5.699312      +5.698821   +0.01552   3.1e-15
   10000      +8.001421      +8.001406   +0.00142   2.1e-14
  100000     +10.304086     +10.303991   +0.02994   3.2e-14
 1000000     +12.606604     +12.606577   +0.02720   1.3e-13
10000000     +14.909169     +14.909162   +0.02386   1.7e-13
H9 : CONFIRMEE


### Lecture du résultat

**H9 : CONFIRMÉE** (*Mesuré*), sur ses cinq points.

- **Représentation.** Pour les nombres premiers $p < 200$ autres que 3, la forme $r^2 - rt + t^2$ représente exactement les $p \equiv 1 \pmod 3$. C'est un théorème classique de représentation par la forme norme de l'anneau des entiers de $\mathbb{Q}(\sqrt{-3})$. Le calcul le vérifie sur un ensemble fini ; il ne le démontre pas.
- **Poids.** Sur les 665 134 puissances de nombres premiers jusqu'à $10^7$, le facteur $1 - \chi(n)$ vaut 2 pour 332 424 d'entre elles, les puissances impaires de nombres premiers inertes ; 1 pour les 14 puissances de 3 ; 0 pour toutes les autres. Aucun écart avec la phrase du guide.
- **Valeur en 1.** Le calcul donne $L(1, \chi) = 0{,}6045997876$, à $4{,}9 \times 10^{-10}$ de la valeur classique $\pi / (3\sqrt{3}) = 0{,}6045997881$. L'écart est compatible avec l'erreur en $h^2$ de la moyenne. Le calcul donne aussi $L'(1, \chi) = 0{,}2227$, donc $L'/L(1, \chi) = 0{,}3683$.
- **Les deux membres.** Rapporté à $a$, l'écart entre $\mathrm{ASK}(a)$ et $\mathrm{MAIN}(a)$ reste inférieur à $6{,}5 \times 10^{-3}$ sur toute la plage : $\mathrm{MAIN}$ est bien le terme principal de $\mathrm{ASK}$. Rapporté à $\sqrt{a}$, cet écart reste entre $0{,}0014$ et $0{,}064$.
- **L'identité.** L'écart entre $\mathrm{ASK}(a) - \mathrm{MAIN}(a)$ et $G(a) - g_\chi(a)$ ne dépasse pas $1{,}7 \times 10^{-13}$ une fois rapporté à $a$. C'est le niveau de l'arrondi flottant accumulé sur des sommes de plusieurs centaines de milliers de termes.

**Ce que ce calcul établit, et ce qu'il n'établit pas.** Avec $w = 1$, l'identité du guide est une tautologie : développer $G - g_\chi$ avec les définitions de la partie B redonne $\mathrm{ASK} - \mathrm{MAIN}$, terme à terme. La dernière colonne contrôle donc l'implémentation ; elle ne découvre rien. La petitesse de la différence est cohérente avec la formule explicite de B.4, étendue aux fonctions $L$ : c'est un fait classique, non vérifié ici. Le contenu du guide est ailleurs : c'est l'**inégalité de pentes**, issue de la géométrie arithmétique, qui borne $(\mathrm{ASK}(a) - \mathrm{MAIN}(a))/b$ par une puissance de $x$ fois $(x/m + m + \sqrt{x})$. Cette inégalité n'est ni démontrée ni testée ici. Et le cas $d = 3$, $w = 1$ n'est pas celui de la preuve : le guide le dit lui-même, la combinaison fait intervenir $\chi_{-d}$ au lieu de $\psi$, avec un signe moins, et ce sont les poids inégaux du § 5 qui la réparent.

*Statut :* les cinq points de H9 sont *Mesurés* ; l'inégalité de pentes et son lien avec $(\star)$ sont *Cités*.

### C.3 — Trop courte ou trop longue : le compromis

L'inégalité précédente dépend de la longueur $m$ de la chaîne de relevés, pour une somme de von Mangoldt de longueur $x$. Le guide (§ 4) énonce ce qu'elle devient une fois appliquée à la chaîne, divisée par le poids commun $b$ et moyennée sur les $m$ relevés :

> « (1/b)(ASK(a) − MAIN(a)) ≤ x^{O(ε)}(x/m + m + √x). »

Puis il explique la forme du membre de droite :

> « On the right, the errors of the whole chain are shared among its m lifts, which gives x/m; each lift has a cost that grows with the length of the chain, which gives m; the third term does not depend on m. A short chain gains too little and a long one costs too much: the optimum is m ≍ √x. In the archive √x δ₁ ≤ m ≤ 2√x δ₁ with δ₁ = x^{−ε/8}. »

Et il nomme l'énoncé de l'archive qui porte ce résultat :

> « In the archive the outcome of this section is a single proposition, TopG: whenever hosts exist on arbitrarily late windows and satisfy the cohomological bound of §6, they carry chains for which the inequality holds, with the exponents hidden in x^{O(ε)} fixed before ε is chosen. »

Ces énoncés sont *Cités*. Les « hôtes » dont parle `TopG` sont reconstruits fenêtre par fenêtre, puisque les nombres premiers auxiliaires en dépendent.

**Pourquoi $m \approx \sqrt{x}$, en une ligne.** À $\sqrt{x}$ et au facteur $x^{O(\varepsilon)}$ près, le membre de droite est $x/m + m$. En posant $u = m / \sqrt{x}$ :
$$\frac{x/m + m}{2\sqrt{x}} = \frac{1}{2}\left(u + \frac{1}{u}\right),$$
une expression minimale, égale à 1, quand $u = 1$. Elle grandit dès que $m$ s'écarte de $\sqrt{x}$, dans un sens ou dans l'autre : une chaîne trop courte partage trop peu l'erreur, une chaîne trop longue paie trop de relevés.

**D'où vient le 9 de $(\star)$ ?** La partie B a laissé ce compte en suspens. Le guide (§ 5) le donne en unités de $\varepsilon$ :

> « In units of ε, the exponent lost is 13/8 in the sizes of m and of the family, 5 in the error terms of the slope inequality, 1 in the density of the thick set, 3/8 in constants, and 1 in removing the twisted characters. The sum is 13/8 + 5 + 1 + 3/8 + 1 = 9, the 9 of (⋆). »

Chacune de ces pertes vient d'une étape de la preuve (§ 5 et § 6), qui n'est ni reproduite ni testée ici. Le calcul qui suit ne porte que sur le compromis $x/m + m$, et sur le prix $x^{9\varepsilon}$ de l'exposant $1/2 + 9\varepsilon$ dans $(\star)$.

**Ce que l'auteur en dit.** Dans sa note, la longueur de chaîne suggérée est motivée par l'imitation de la multiplicité (*Annoncé*) :

> « [·]'s suggested length of chain of theta lifts, in view of each mimicking multiplicity, was √X. (And yet, what else could it be?) »

**Le calcul.** Trois échelles, $x = 10^6$, $10^9$ et $10^{12}$. D'abord le coût relatif $\frac{1}{2}(u + 1/u)$ quand $m$ s'écarte de $\sqrt{x}$ d'un facteur $x^{\pm 0{,}1}$, et le minimum exact sur les entiers. Puis les deux bornes de l'intervalle de l'archive, pour $\varepsilon = 10^{-2}$ et $10^{-3}$. Le calcul est purement arithmétique : il ne fait appel à aucune des sommes des parties A et B.

**H10 — Hypothèse pré-enregistrée.** *Pour $x = 10^6$, $10^9$ et $10^{12}$, le minimum de $x/m + m$ sur les entiers $m \ge 1$ sera atteint en un entier à distance au plus 1 de $\sqrt{x}$.*

*C'est un contrôle : l'inégalité des moyennes le garantit déjà. Il vérifie que le programme calcule bien ce qu'il annonce.*

In [18]:
def cout_relatif(x, m):
    """Cout relatif (x / m + m) / (2 sqrt(x)) du membre de droite de l inegalite : il vaut 1 quand m = sqrt(x)."""
    return (x / m + m) / (2 * sqrt(x))

print('     x    m = sqrt(x) x^-0.1   m = sqrt(x)   m = sqrt(x) x^0.1   minimiseur entier   sqrt(x)')
h10 = True
for x in (10 ** 6, 10 ** 9, 10 ** 12):
    rac = sqrt(x)
    couts = [cout_relatif(x, rac * x ** e) for e in (-0.1, 0.0, 0.1)]        # m s ecarte de sqrt(x) d un facteur x^e
    meilleur = min(range(1, 2 * int(rac) + 2), key=lambda m: x / m + m)      # minimum sur les entiers
    h10 = h10 and abs(meilleur - rac) <= 1                                  # le minimiseur est a distance <= 1 de sqrt(x)
    print(f'{x:>8.0e}   {couts[0]:>16.4f}   {couts[1]:>10.4f}   {couts[2]:>14.4f}   {meilleur:>15}   {rac:.1f}')
print('H10 :', 'CONFIRMEE' if h10 else 'REFUTEE')

# L intervalle de l archive : sqrt(x) delta_1 <= m <= 2 sqrt(x) delta_1, avec delta_1 = x^(-eps / 8).
# On evalue le cout relatif aux deux bornes, puis le facteur x^(9 eps) de l exposant 1/2 + 9 eps de l enonce (star).
print("intervalle de l archive : sqrt(x) delta_1 <= m <= 2 sqrt(x) delta_1, delta_1 = x^(-eps / 8)")
for eps in (0.01, 0.001):
    for x in (10 ** 6, 10 ** 9, 10 ** 12):
        rac = sqrt(x)
        delta1 = x ** (-eps / 8)
        couts = [cout_relatif(x, rac * delta1), cout_relatif(x, 2 * rac * delta1)]   # borne basse, puis borne haute
        print(f'  eps = {eps}, x = {x:.0e} : delta_1 = {delta1:.5f}, cout relatif de x/m + m entre {min(couts):.4f} et {max(couts):.4f}, facteur x^(9 eps) = {x ** (9 * eps):.3g}')

     x    m = sqrt(x) x^-0.1   m = sqrt(x)   m = sqrt(x) x^0.1   minimiseur entier   sqrt(x)
   1e+06             2.1161       1.0000           2.1161              1000   1000.0
   1e+09             4.0346       1.0000           4.0346             31623   31622.8


   1e+12             7.9560       1.0000           7.9560           1000000   1000000.0
H10 : CONFIRMEE
intervalle de l archive : sqrt(x) delta_1 <= m <= 2 sqrt(x) delta_1, delta_1 = x^(-eps / 8)
  eps = 0.01, x = 1e+06 : delta_1 = 0.98288, cout relatif de x/m + m entre 1.0001 et 1.2372, facteur x^(9 eps) = 3.47
  eps = 0.01, x = 1e+09 : delta_1 = 0.97443, cout relatif de x/m + m entre 1.0003 et 1.2310, facteur x^(9 eps) = 6.46
  eps = 0.01, x = 1e+12 : delta_1 = 0.96605, cout relatif de x/m + m entre 1.0006 et 1.2248, facteur x^(9 eps) = 12
  eps = 0.001, x = 1e+06 : delta_1 = 0.99827, cout relatif de x/m + m entre 1.0000 et 1.2487, facteur x^(9 eps) = 1.13
  eps = 0.001, x = 1e+09 : delta_1 = 0.99741, cout relatif de x/m + m entre 1.0000 et 1.2481, facteur x^(9 eps) = 1.21
  eps = 0.001, x = 1e+12 : delta_1 = 0.99655, cout relatif de x/m + m entre 1.0000 et 1.2474, facteur x^(9 eps) = 1.28


### Lecture du résultat

**H10 : CONFIRMÉE** (*Mesuré*). Pour $x = 10^6$, $10^9$ et $10^{12}$, le minimum de $x/m + m$ sur les entiers est atteint en $m = 1000$, $31\,623$ et $1\,000\,000$ : l'écart à $\sqrt{x}$ vaut $0$, $0{,}2$ et $0$. Le programme calcule ce qu'il annonce.

- **Le compromis se lit sur le premier tableau.** Quand $m$ s'écarte de $\sqrt{x}$ d'un facteur $u = x^{\pm 0{,}1}$, soit $u \approx 3{,}98$, $7{,}94$ et $15{,}85$ à ces trois échelles, le coût relatif passe de $1{,}0000$ à $2{,}1161$, puis $4{,}0346$, puis $7{,}9560$. Il croît comme $u/2$ dès que $u$ dépasse quelques unités : c'est la formule de l'introduction, $\frac{1}{2}(u + 1/u)$. Une chaîne dix fois trop longue coûte $(10 + 0{,}1)/2 \approx 5$ fois le prix du minimum ; une chaîne dix fois trop courte, autant.
- **L'intervalle de l'archive est peu coûteux.** À sa borne basse ($m = \sqrt{x}\,\delta_1$), le coût relatif reste entre $1{,}0000$ et $1{,}0006$ ; à sa borne haute ($m = 2\sqrt{x}\,\delta_1$), entre $1{,}2248$ et $1{,}2487$ : de $22$ à $25\,\%$ de plus qu'au minimum.
- **Le facteur $x^{9\varepsilon}$, lui, n'est pas petit.** Il vaut $3{,}47$, $6{,}46$ et $12$ pour $\varepsilon = 10^{-2}$, et $1{,}13$, $1{,}21$ et $1{,}28$ pour $\varepsilon = 10^{-3}$.

**Ce que ce calcul établit, et ce qu'il n'établit pas.** L'arithmétique du compromis est exacte, et c'est tout ce que les tableaux mesurent. Le 9 de $(\star)$ est *Cité* : le carnet affiche le facteur $x^{9\varepsilon}$, il ne recalcule pas les pertes que le guide y additionne. Quant à l'inégalité de pentes, celle dont le membre de droite porte ce compromis, elle n'est ni démontrée ni testée ici.

## Limites et bilan honnête

**Ce que le carnet établit** (*Mesuré*), sur des objets assez petits pour que tout se calcule :

- **Le budget d'une fonction sur une courbe.** Une fonction non nulle a au plus autant de zéros, comptés avec multiplicité, que l'ordre de son pôle ; sur les 68 cas calculés, le déterminant de la matrice des dérivées est égal à sa forme close.
- **L'ordre de grandeur des écarts.** Sur les grilles mesurées, $10^3 \le x \le 10^7$, $\psi(x) - x$ vaut en valeur absolue au plus $0{,}46\,\sqrt{x}$, et $G(y) - \log 2\pi$ au plus $0{,}023\,\sqrt{y}$ : une fraction de $\sqrt{x}$, de signe changeant, sans que la fenêtre permette de conclure sur une limite.
- **Ce qu'une majoration sur des fenêtres interdit.** La formule explicite tronquée aux cent premiers zéros rend compte de $G(y) - \log 2\pi$ à mieux que $4 \times 10^{-4}\sqrt{y}$ sur les fenêtres testées ; si le premier de ces zéros était déplacé en $0{,}9 + i\gamma_1$, le maximum de $|G(y) - \log 2\pi|/y^{0{,}6}$ serait multiplié par $7{,}5$ entre la fenêtre $Y = 10^2$ et la fenêtre $Y = 10^4$, contre $0{,}52$ avec les vrais zéros.
- **L'arithmétique du compromis.** Le membre de droite $x/m + m$ de l'inégalité du guide est minimal en $m \approx \sqrt{x}$, et l'écart se paie en $\frac{1}{2}(u + 1/u)$.

**Ce que ces mesures ne prouvent pas.**

- **Rien sur l'hypothèse de Riemann.** Un écart petit sur une fenêtre finie est une observation numérique : elle est compatible avec l'hypothèse, et aussi avec sa négation par un zéro fautif d'ordonnée élevée, qui ne se verrait que sur des fenêtres plus grandes (le calcul de B.5 donne l'ordre de grandeur du seuil).
- **Aucune étape de l'argument annoncé.** La chaîne de relevés thêta, la tour cyclotomique, les nombres premiers auxiliaires et la théorie d'Iwasawa sont *Cités*. L'inégalité de pentes n'est pas testée, et le cas calculé en C.2, avec $d = 3$ et un poids $1$, n'est pas un cas de la preuve : le guide demande notamment à $d$ d'être congru à $7$ modulo $16$.
- **Rien sur l'archive de preuves.** Ce carnet ne la lit pas. Ce qui en est dit vient d'un dénombrement fait hors du carnet (*Rapporté*, voir Raccords au dépôt) : ni compilation, ni rejeu du noyau, ni lecture des énoncés.

## Références

Toutes les sources sont lues depuis leur copie archivée au gisement partagé (GDrive `MyIA/IA/Bibliographie IA`, rayon `NumberTheory`) ; leur statut est précisé en tête de carnet.

- L. Alpöge, *Remark on the Riemann Hypothesis*, note du 20 novembre 2026, 4 pages (*Annoncé*).
- *The proof of the Riemann hypothesis announced by L. Alpöge — Its architecture: the statements, the flow, and why each idea calls for the next*, publié sous le nom de projet `riemann-rev`, 11 pages, relevés arrêtés au 20 novembre 2026 à 23 h 51 UTC (*Cité*).
- *Lean verification of the Riemann Hypothesis*, déclaration conjointe de l'équipe de développement de Lean et de l'initiative Mathlib, 21 novembre 2026 (*Cité*).
- L'issue [#27589](https://github.com/jsboige/CoursIA/issues/27589) du dépôt : sources, statuts et dénombrement de l'archive (*Rapporté*).
- Le théorème de Hasse, la formule d'Euler–Maclaurin, la fonction $Z$ de Hardy et la formule explicite sont des résultats classiques : le carnet les utilise sans les démontrer et ne les rattache à aucune édition.

## Raccords au dépôt

- **[11-riemann-roch](11-riemann-roch.ipynb)** : le théorème qui, sur une courbe, compte les fonctions à pôles prescrits. La partie A met en œuvre, sans l'invoquer, la règle qui le prépare : une fonction a au plus autant de zéros que l'ordre de ses pôles.
- **[02-formule-explicite](02-formule-explicite.ipynb)** : la formule explicite, que la partie B écrit pour $G$ puis tronque aux cent premiers zéros.
- **[12-zeros-simples-ligne-critique](12-zeros-simples-ligne-critique.ipynb)** : le carnet dont celui-ci est l'accrétion ; ses sorties donnent les premiers zéros et le centième, que la partie B retrouve par un autre calcul.
- **[13-bernoulli-zeta-negatifs-todd](13-bernoulli-zeta-negatifs-todd.ipynb)** : les nombres de Bernoulli, dont la partie B.0 se sert pour la formule d'Euler–Maclaurin.
- **[14-deux-annonces-grh-meme-jour](14-deux-annonces-grh-meme-jour.ipynb)** : lire une annonce de démonstration sans la tenir pour une preuve, posture dont ce carnet part.
- **[Serre100/01-corps-finis-borne-hasse](../Serre100/01-corps-finis-borne-hasse.ipynb)** (autre série) : la borne de Hasse, distillée en un calcul, là où la partie A se contente de la mesurer.
- **Dans Mathlib** (*Rapporté*) : relevé du 20 novembre 2026, consigné sur l'issue #27589. L'énoncé `RiemannHypothesis` est défini dans `Mathlib/NumberTheory/LSeries/RiemannZeta.lean`, ligne 185 à la révision que fixe l'archive (commit `0df444a3`) : tout $s$ tel que $\zeta(s) = 0$, qui n'est pas un zéro trivial et diffère de $1$, a pour partie réelle $1/2$. L'archive annoncée déclare, dans `lean/src/Main.lean`, un terme de cet énoncé (`theorem riemannHypothesis : RiemannHypothesis := rhOfAll`) ; le dénombrement de l'archive (239 275 554 octets ; 175 187 modules `.lean`) n'y a trouvé, par recherche de texte, aucun `axiom`, `sorry`, `admit` ni `native_decide`. Ce relevé n'est ni une compilation ni un rejeu du noyau : il ne dit pas que la preuve est juste.

## À retenir

1. **La racine carrée vient d'un budget.** Sur une courbe, une fonction non nulle a au plus autant de zéros, comptés avec multiplicité, que l'ordre de son pôle. L'argument de Stepanov fabrique une fonction qui s'annule en chaque point du corps fini : compter ses zéros borne le nombre de points, autour de $p + 1$ et à $2\sqrt{p}$ près.
2. **Les sommes de von Mangoldt oscillent autour de leur terme principal.** $\psi(x)$ autour de $x$, $G(y)$ autour de $\log 2\pi$, d'une fraction de $\sqrt{x}$ sur les fenêtres mesurées ; la formule explicite rend compte de ces oscillations par les zéros de $\zeta$.
3. **Une majoration sur des fenêtres a un contenu.** Elle interdit la croissance qu'un zéro hors de la ligne critique imprimerait à l'écart : c'est ce que montre l'expérience de B.5.
4. **Une longueur de chaîne est un compromis.** Un terme en $x/m$, un terme en $m$ : le minimum est en $m \approx \sqrt{x}$, et c'est de là que vient, d'après le guide, la racine carrée de l'énoncé $(\star)$.

Et une consigne de lecture : rien de ce que le carnet nomme *Cité*, *Annoncé* ou *Rapporté* n'a été vérifié ici.

## Vérifiez votre compréhension

1. Exercice 1 : sur $\mathbb{F}_{13}$, avec les points $(6, 4)$ et $(2, 5)$ et $k = 2$, que valent le déterminant de la matrice des dérivées et sa forme close ?
2. Pourquoi la règle « au plus autant de zéros que l'ordre du pôle » donne-t-elle une majoration du nombre de points d'une courbe ?
3. Exercice 2 : en $x = 5 \times 10^6$, que valent $\psi(x) - x$ et son rapport à $\sqrt{x}$ ?
4. Si l'on remplace le premier zéro de $\zeta$ par $0{,}9 + i\gamma_1$, que devient le maximum de $|G(y) - \log 2\pi|/y^{0{,}6}$ entre la fenêtre $Y = 10^2$ et la fenêtre $Y = 10^4$ ?
5. Exercice 3 : combien de fois la fonction $Z$ change-t-elle de signe entre $20$ et $30$ ?
6. Exercice 4 : par combien de zéros le nombre $2019!$ finit-il ?
7. Pourquoi la longueur $m$ de la chaîne est-elle choisie proche de $\sqrt{x}$ ?

<details>
<summary>Réponses</summary>

1. Déterminant $12$, forme close $12$.
2. On construit une fonction non nulle qui s'annule en chaque point cherché. Si son pôle est d'ordre $D$, elle a au plus $D$ zéros, comptés avec multiplicité : le nombre de points, multiplié par l'ordre d'annulation, est au plus $D$.
3. $\psi(x) - x = +971{,}14$, soit $+0{,}434$ fois $\sqrt{x}$.
4. Il est multiplié par $7{,}5$ dans le calcul de B.5, contre $0{,}52$ avec les vrais zéros : l'écart n'est plus d'ordre $\sqrt{y}$.
5. Deux fois.
6. Par $502$ zéros : $v_5(2019!) = 502$ et $v_2(2019!) = 2011$.
7. Parce que le membre de droite $x/m + m$ est minimal quand ses deux termes sont égaux, c'est-à-dire pour $m = \sqrt{x}$ ; à un facteur $u$ près, il est multiplié par $\frac{1}{2}(u + 1/u)$.

</details>

## Pour aller plus loin

- **Carnet précédent** : [12-zeros-simples-ligne-critique](12-zeros-simples-ligne-critique.ipynb) — deux tiers des zéros de zêta simples et sur la ligne critique.
- **Carnet suivant** : [13-bernoulli-zeta-negatifs-todd](13-bernoulli-zeta-negatifs-todd.ipynb) — les valeurs de zêta aux entiers négatifs.
- **Dans la même série** : [02-formule-explicite](02-formule-explicite.ipynb) (la formule explicite) · [11-riemann-roch](11-riemann-roch.ipynb) (compter les fonctions d'une courbe) · [14-deux-annonces-grh-meme-jour](14-deux-annonces-grh-meme-jour.ipynb) (lire une annonce).
- **Carnet voisin d'une autre série** : [01-corps-finis-borne-hasse](../Serre100/01-corps-finis-borne-hasse.ipynb) — la borne de Hasse, du côté des corps finis.
- **Frontière Lean** : aucun fichier de [`riemann200_lean`](riemann200_lean/Riemann200/LiminfForme.lean) ne formalise ce carnet ; ses deux modules, [`LiminfForme.lean`](riemann200_lean/Riemann200/LiminfForme.lean) et [`Memoire1859.lean`](riemann200_lean/Riemann200/Memoire1859.lean), disent dans leur en-tête ne pas prouver `RiemannHypothesis`.